# Predicción del planeta c a partir de la estrella y el planeta b

Selecciona un sistema y el planeta **b**. La distancia de **c** en UA y su masa
opcional en masas terrestres se usan para evaluar una región en log₁₀(a) y log₁₀(m).

- **referencia** ajusta un KDE ponderado de la clase espectral, excluyendo todo el
  sistema objetivo. El peso predeterminado es A_est / (A_est + A_inest + A_col_m1 + A_col_m2).
  Cada isla corresponde a una componente que contiene su pico. Los cruces de su nivel
  se interpolan dentro de las celdas. El bootstrap remuestrea sistemas completos.
- **solo_b** coloca una gaussiana en b. El peso se cancela al normalizar a 100 %.
  Se hereda el kernel del mismo tipo o de todos los tipos disponibles, siempre
  excluyendo el sistema objetivo. Si falta una muestra apta, se advierte y se usa
  el ancho de PRED. En el nivel 90,9 %, el semiancho radial es 0,437·σ_log(a)
  (0,1529 dex cuando σ_log(a)=0,35 dex).
- **ambos** ejecuta las dos evaluaciones.

Con masa, la decisión nominal comprueba la pertenencia al contorno 2D y a su
proyección radial. Sin masa, usa la proyección radial. Los intervalos 1σ y 3σ
son envolventes marginales de los bordes. En solo_b, las fuentes de dispersión
se identifican por separado. La cobertura describe la fracción del rango observado
en log₁₀(a), bajo una línea base uniforme en ese rango.

La celda de ejecución permite usar el diálogo original o configurar PRED con
entrada por parámetros. La última celda contiene pruebas autocontenidas.

In [ ]:
import os
import re
import csv
import tempfile
from pathlib import Path
from collections import Counter
import math
import pickle
import hashlib
import itertools
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize
from matplotlib.lines import Line2D
from matplotlib.patches import Rectangle
from scipy.ndimage import maximum_filter, label as _label_cc
from scipy.stats import gaussian_kde
from scipy.optimize import linear_sum_assignment
from scipy.interpolate import RegularGridInterpolator
try:
    import tkinter as tk
    from tkinter import filedialog, simpledialog, messagebox
except ImportError:   # sin tkinter (servidor): usa PRED['entrada'] = 'parametros'
    tk = filedialog = simpledialog = messagebox = None

try:
    from tqdm import tqdm  # pip install tqdm
except ImportError:
    def tqdm(iterable, total=None, desc=None, **kwargs):
        """
        Fallback mínimo si la librería 'tqdm' no está instalada: imprime el
        progreso como porcentaje sobre la misma línea de la terminal.
        """
        if total is None:
            try: 
                total = len(iterable)
            except TypeError:
                total = None
        prefijo = f"{desc}: " if desc else ""
        for i, item in enumerate(iterable, start=1):
            yield item
            if total:
                pct = 100 * i / total
                print(f"\r{prefijo}{pct:5.1f}% ({i}/{total})", end="", flush=True)
        if total:
            print()

Configuración

In [ ]:
# ---------------------------------------------------------------------------
# PARÁMETROS GLOBALES Y UTILIDADES
# ---------------------------------------------------------------------------
N_BOOTSTRAP_INCERTIDUMBRE = 1000  # número de réplicas bootstrap por panel
KDE_BW = 0.55                     # factor de ancho nominal y de bootstrap adaptable
KDE_GRID_RES = 150j
BOOTSTRAP_GRID_RES = 250j          # malla del bootstrap

# ---- pesos del KDE ---------------------------------------------------------
#   "area"     : log10(A_estable + 1)
#   "fraccion" : A_est / (A_est + A_inest + A_colisión m2)   <- usa las tres regiones
#   "neto"     : log10(max(A_est - (A_inest + A_col), 0) + 1)
MODO_PESO_DEFECTO = "fraccion"

# ---- incertidumbre 1σ / 2σ / 3σ ---------------------------------------------
SIGMAS = (1, 2, 3)                 # k-sigma que se calculan y exportan a CSV
SIGMAS_DIBUJAR = (1, 2, 3)         # k-sigma que se dibujan como bandas anidadas
METODO_SIGMA_DIBUJO = "percentil"
ALPHA_BANDA_SIGMA = 0.07           
DIBUJAR_BANDAS_SIGMA = False       
GAP_REGIONES_DEX = 0.08
COLOR_LIMITES = "lime"           

COLORES_REGIONES = ["lime", "red", "dodgerblue", "magenta"]
# (estilo a_min, estilo a_max) por región; el estilo distingue la región
LS_REGIONES = [("-", "--"),
               ("-.", ":"),
               ((0, (5, 1.5)), (0, (1, 1))),
               ((0, (3, 1, 1, 1, 1, 1)), (0, (5, 5)))]


BOOT_POR_SISTEMAS = True      
KDE_COV_FIJA = True          
METODO_ERROR_REGION = "percentil"   
MODO_UMBRAL_REGION = "pico"   
                              
MASA_HDR = 0.50
LIM_SISTEMA_SOLAR_X = (-1.0, 2.0)   # log10(distancia / AU)
MARCA_X_SISTEMA_SOLAR = math.log10(46.0)   # log10(AU) de la región a resaltar: 46 AU (None = sin marca)
LIM_SISTEMA_SOLAR_Y = None          # log10(masa / M_Tierra)

# ---- regiones estables como ISLAS (picos locales) y su incertidumbre ----------
# "picos"  : cada isla de densidad (máximo local >= NIVEL_MIN_PICO %) es una región.
# "umbral" : método anterior (todo >= NIVEL_REGION_ESTABLE % del máximo global).
MODO_REGIONES = "picos"
NIVEL_MIN_PICO = 20.0          
NIVEL_MIN_PICO_SOLAR = 10.0    
PICO_VENTANA_DEX = 0.4        
TOL_ASIGNACION_DEX = 0.5       
TOL_ASIGNACION_Y_DEX = 1.0     
ERROR_ASIMETRICO = True       

COLORES_BANDAS_REGION = ["white", "cyan", "gold", "orchid"]

NIVELES_POR_ALTURA = True

# [NUEVO] Número de regiones (islas de mayor densidad) por figura. None / no listado = todas las que
# detecte el método. Figura en fila (Spectral Type X):
REGIONES_FILA = {"F": 2,       # isla de ~1 M_Tierra + isla de masas neptunianas
                 "G": 2,       # región principal (90.9 %) + isla cercana a Júpiter
                 "K": 3,
                 "M": 1}       # solo la región acotada al 90.9 %
# Figuras por multiplicidad: (tipo, nº de planetas) -> nº de regiones
REGIONES_MULT = {("F", 3): 3,
                 ("K", 3): 3, ("K", 4): 3, ("K", 5): 1,
                 ("M", 3): 1, ("M", 4): 1, ("M", 5): 1, ("M", 7): 1}

DIBUJAR_ISOLINEAS = True           # isolíneas blancas difuminadas (False = sin isolíneas)

# ---- límites de masa de cada región (líneas horizontales / "cuadrados de masa") ---------------
# Para cada región se dibuja el RECTÁNGULO de la isla: las líneas verticales a_min / a_max (de altura
# completa) y las horizontales m_min / m_max entre a_min y a_max, con un relleno tenue del color de la región.
DIBUJAR_LIMITES_MASA = True
LIMITES_MASA_ANCHO_COMPLETO = False   # False: la línea cubre solo [a_min, a_max]; True: cruza todo el panel
RELLENO_CAJA_MASA = 0.08              # alpha del relleno del rectángulo de cada región (0 = sin relleno)

# ---- salida y ejecución --------------------------------------------------------
SALIDA_CARPETA = "graficos_magma_con_sistemas_azules"
MOSTRAR_FIGURAS = True             # False = solo guarda (útil sin pantalla)

# ---- caché del KDE ------------------------------------------------------------
# El KDE + las regiones + el bootstrap de cada panel se guardan en <salida>/cache_kde/*.pkl.
# La clave del archivo es un hash de los datos del panel y de TODOS los parámetros que afectan
# al resultado: si cambias datos, pesos, ancho de banda, niveles, etc., se recalcula solo.
USAR_CACHE_KDE = True
FORZAR_RECALCULO_KDE = False       # True = ignora la caché y la sobrescribe
CACHE_KDE_NOMBRE = "cache_kde"
SEMILLA_BOOTSTRAP = 12345          # semilla fija -> resultados reproducibles (None = aleatoria)
_CACHE_VERSION = "v6"              # bordes subcelda, bootstrap por sistemas y metadatos
_CACHE_DIR = None

# ---- leyendas (tamaño y posición) ---------------------------------------------
# fs: fuente | ms: marcadores Tierra/Júpiter | ncol: columnas | y: posición vertical bajo el eje
# fs_cajas: cajas de a_min / a_max sobre el panel
LEYENDA_DEFECTO = {"fs": 9.0, "ms": 7, "ncol": 2, "y": -0.16, "fs_cajas": 8.5}
LEYENDA_POR_TIPO = {                # ajustes por tipo espectral (se suman a LEYENDA_DEFECTO)
    "F": {"fs": 9.5, "ncol": 2, "y": -0.18},
}
LEYENDA_SOLAR = {"fs": 12, "ms": 8, "ncol": 3, "y": -0.15, "fs_cajas": 10}
LEYENDA_COMPARACION_FS = 10.5      # leyenda de la figura Sistema Solar vs. G
ANCHO_PANEL_TIPO = 7.4             # ancho (pulgadas) por panel en plot_figuras_por_tipo
ANCHO_PANEL_MATRIZ = 7.2           # ídem en plot_matriz_tipo_multiplicidad
ANCHO_PANEL_FILA = 7.6             # ídem en plot_tipos_espectrales_fila

# ---- Sistema Solar -----------------------------------------------------------------
# Ya NO se mezcla con las estrellas tipo G: las figuras F, G, K, M se hacen sin él y el
# Sistema Solar tiene su propia figura unificada (etapa "solar").
SISTEMA_SOLAR = {
    "activo": True,
    "preguntar": True,             # True = pregunta con un diálogo si lo incluye
    "tipo_estrella": "G",          # solo se usa en la etapa "comparar" (Sistema Solar vs. G)
    "nombre_sistema": "Solar System",
    "n_planetas_total": 8,
    "marcador": "*", "color": "gold",
    # nombre: (a [UA], masa [M_tierra], A_estable, A_inestable, A_colisión_m2 [km²],
    #          zonas_estables, zonas_inestables, zonas_colisión_m2)   (colisión m1 = 0)
    "planetas": {
        "Mercury": (0.387, 0.0553, 3.01e12, 5.33e14, 2.15e12,  56, 9904, 40),
        "Venus":   (0.723, 0.815,  1.27e13, 1.85e15, 4.29e12,  68, 9909, 23),
        "Earth":   (1.000, 1.000,  2.59e13, 3.57e15, 7.92e12,  72, 9906, 22),
        "Mars":    (1.524, 0.107,  5.16e13, 8.24e15, 2.58e13,  62, 9907, 31),
        "Jupiter": (5.203, 317.8,  6.82e15, 9.00e16, 9.68e12, 704, 9295,  1),
        "Saturn":  (9.537, 95.16,  1.29e16, 3.13e17, 2.61e14, 397, 9595,  8),
        "Uranus":  (19.19, 14.54,  2.03e16, 1.30e18, 1.85e15, 154, 9832, 14),
        "Neptune": (30.07, 17.15,  5.57e16, 3.18e18, 4.53e15, 172, 9814, 14),
    },
}

Parámetros de la predicción de c

In [ ]:
# ---------------------------------------------------------------------------
# PARÁMETROS DE LA PREDICCIÓN DE c
# ---------------------------------------------------------------------------
PRED = {
    # "dialogo": ventanas para elegir carpetas, sistema, planeta b y distancia de c.
    # "parametros": usa los valores de más abajo (sin ventanas; útil en servidores sin pantalla).
    "entrada": "dialogo",

    # "referencia" | "solo_b" | "ambos"   (ver la celda de introducción)
    "modo_region": "solo_b",

    # Pesos con las 4 áreas: A_est / (A_est + A_inest + A_col_m1 + A_col_m2)
    "modo_peso": "fraccion4",

    "n_bootstrap": 1000,           # réplicas de bootstrap o Monte Carlo; 0 desactiva el remuestreo
    "nivel_min_pico": None,        # None = NIVEL_MIN_PICO

    # Ancho de kernel (dex en log a, dex en log m) si no hay muestra de referencia para heredarlo (solo_b)
    "cov_solo_b_dex": (0.35, 0.80),

    # None = incertidumbre observacional desconocida; cero es explícito.
    "sigma_a_b": None,              # UA
    "sigma_m_b": None,              # masas terrestres
    "sigma_log_ancho": math.log(2.0) / 1.96,  # sensibilidad: ~95% de factores entre 0.5× y 2×
    "catalogo_multiplicidad": None, # None busca la tabla local de estrellas; admite ruta explícita

    # ---- solo con entrada = "parametros" ----
    "carpetas": [],                # [("G", "ruta/Resultados_Estrellas_G"), ("K", "ruta/..."), ...]
    "sistema": None,               # nombre del sistema según README (ver datos_extraidos.csv)
    "planeta_b": None,             # nombre según README o sufijo orbital exacto, por ejemplo "b"
    "a_c": None,                   # semi-eje de c en UA
    "m_c": None,                   # masa de c en M_Tierra (opcional)

    "salida": None,                # None = <primera carpeta>/graficos_prediccion_c
}

Pesos estadísticos (ahora con las 4 áreas: `fraccion4`)

In [ ]:
def _mostrar_o_cerrar(fig):
    if MOSTRAR_FIGURAS:
        plt.show()
    plt.close(fig)


def _safe_log10(x):
    """log10(1 + x) para áreas no negativas, sin perder valores pequeños."""
    return np.log1p(np.asarray(x, dtype=float)) / math.log(10.0)


def _calcular_pesos(df_sub: pd.DataFrame, modo=None) -> np.ndarray:
    """Un área requerida inválida da peso NaN. Se mantienen las cuatro fórmulas."""
    modo = modo or MODO_PESO_DEFECTO
    columnas = {
        "area": ("Area_Estable_km2",),
        "fraccion": ("Area_Estable_km2", "Area_Inestable", "Area_Colision_m2"),
        "fraccion4": ("Area_Estable_km2", "Area_Inestable", "Area_Colision_m1", "Area_Colision_m2"),
        "neto": ("Area_Estable_km2", "Area_Inestable", "Area_Colision_m1", "Area_Colision_m2"),
    }
    if modo not in columnas:
        raise ValueError(f"modo_peso desconocido: {modo!r}")
    areas = df_sub.reindex(columns=columnas[modo]).apply(pd.to_numeric, errors="coerce").to_numpy(dtype=float)
    validas = np.all(np.isfinite(areas) & (areas >= 0), axis=1)
    w = np.full(len(df_sub), np.nan)
    a = areas[validas]
    if modo == "area":
        w[validas] = _safe_log10(a[:, 0])
    elif modo == "neto":
        with np.errstate(over="ignore"):
            w[validas] = _safe_log10(np.maximum(a[:, 0] - a[:, 1:].sum(axis=1), 0.0))
    else:
        # Escalar cada fila antes de sumar evita desbordamiento y conserva el cociente.
        escala = a.max(axis=1, keepdims=True)
        a = np.divide(a, escala, out=np.zeros_like(a), where=escala > 0)
        total = a.sum(axis=1)
        w[validas] = np.divide(a[:, 0], total, out=np.zeros_like(total), where=total > 0)
    return w


def _pesos_validos(w) -> bool:
    w = np.asarray(w, dtype=float)
    return bool(w.ndim == 1 and np.isfinite(w).all() and (w >= 0).all() and (w > 0).any())


def _desde_log10(v):
    # Un borde puede exceder el rango lineal de float aunque sea representable en dex.
    with np.errstate(over="ignore", under="ignore", invalid="ignore"):
        p = float(np.power(10.0, v))
    return p if p > 0 and math.isfinite(p) else np.nan


def _validar_malla(xlims, ylims):
    for nombre, limites in (("xlims", xlims), ("ylims", ylims)):
        if len(limites) != 2 or not all(np.isfinite(limites)) or limites[0] >= limites[1]:
            raise ValueError(f"{nombre} debe contener dos límites finitos crecientes.")


def _n_replicas(n, defecto):
    n = defecto if n is None else n
    if isinstance(n, (bool, np.bool_)) or not isinstance(n, (int, np.integer)) or n < 0:
        raise ValueError("n_bootstrap debe ser un entero no negativo.")
    return int(n)


def _normalizar_campo(Z):
    maximo = float(np.max(Z))
    return 100.0 * (Z / maximo) if np.isfinite(Z).all() and maximo > 0 else None


def _datos_kde(df):
    cols = ["Distancia_UA", "Masa_Tierra"]
    vals = df[cols].apply(pd.to_numeric, errors="coerce")
    valido = (np.isfinite(vals) & (vals > 0)).all(axis=1)
    d = df.loc[valido].copy()
    d[cols] = vals.loc[valido].astype(float)
    return d


def _ajustar_kde(x, y, weights, bw=KDE_BW):
    """Ajusta solo puntos con peso positivo y rango suficiente para 2D."""
    if not _pesos_validos(weights):
        return None
    idx = weights > 0
    coords = np.vstack([x[idx], y[idx]])
    if coords.shape[1] < 3 or np.linalg.matrix_rank(coords - coords[:, :1]) < 2:
        return None
    try:
        return gaussian_kde(coords, weights=weights[idx] / weights[idx].max(), bw_method=bw)
    except (np.linalg.LinAlgError, ValueError):
        return None

Estilo de marcadores

In [ ]:
def _asignar_estilos_sistema(df: pd.DataFrame) -> dict:
    """
    Genera combinaciones únicas de marcador/color para cada sistema estelar.
    Devuelve un dict {nombre_sistema: {'marker': ..., 'color': ...}}.
    El Sistema Solar siempre usa su estilo propio (estrella dorada).
    """
    if "Sistema" not in df.columns:
        df = df.copy()
        df["Sistema"] = df["Planeta"].apply(lambda x: str(x).rsplit(" ", 1)[0])

    sistemas_unicos = df["Sistema"].unique()
    marcadores_base = ["o", "s", "^", "D", "v", "p", "*", "h",
                       "H", "X", "d", "P", "<", ">"]
    colores_base = plt.cm.tab20.colors
    combinaciones = itertools.cycle(itertools.product(marcadores_base, colores_base))

    estilos = {}
    for sist, (marker, color) in zip(sistemas_unicos, combinaciones):
        estilos[sist] = {"marker": marker, "color": color}

    ss = SISTEMA_SOLAR
    if ss["activo"] and ss["nombre_sistema"] in estilos:
        estilos[ss["nombre_sistema"]] = {"marker": ss["marcador"], "color": ss["color"]}
    return estilos

In [ ]:
def _calcular_limites_criticos(Z_pct: np.ndarray, X: np.ndarray):
    """Límites del umbral histórico de percentiles."""
    Z_flat = Z_pct.ravel()
    if not Z_flat.size or not np.isfinite(Z_flat).all() or not (Z_flat.max() > 0):
        return None, None
    niveles = np.linspace(np.percentile(Z_flat, 10), np.percentile(Z_flat, 98), 25)
    return _limites_por_nivel(Z_pct, X, niveles[-4])


def _bootstrap_limites_criticos(x, y, weights, xlims, ylims,
                                n_bootstrap=None, grid_res=None, rng=None):
    """
    Estima la incertidumbre asociada al modelo (KDE) en la determinación de
    las distancias críticas a_min y a_max mediante bootstrap no paramétrico.
    """
    n_bootstrap = _n_replicas(n_bootstrap, N_BOOTSTRAP_INCERTIDUMBRE)
    grid_res = grid_res or BOOTSTRAP_GRID_RES
    if rng is None:
        rng = np.random.default_rng()

    n = len(x)
    log_min_boot, log_max_boot = [], []

    if n == 0 or n_bootstrap == 0:
        return log_min_boot, log_max_boot
    _validar_malla(xlims, ylims)
    Xb, Yb = np.mgrid[xlims[0]:xlims[1]:grid_res, ylims[0]:ylims[1]:grid_res]
    positions_b = np.vstack([Xb.ravel(), Yb.ravel()])

    for _ in tqdm(range(n_bootstrap), desc="   Bootstrap a_min/a_max", leave=False, unit="rep"):
        idx = rng.integers(0, n, size=n)
        x_b, y_b, w_b = x[idx], y[idx], weights[idx]

        kde_b = _ajustar_kde(x_b, y_b, w_b, bw=0.6)
        if kde_b is None:
            continue
        Z_b = _normalizar_campo(kde_b(positions_b).reshape(Xb.shape))
        if Z_b is None:
            continue

        log_min_b, log_max_b = _calcular_limites_criticos(Z_b, Xb)
        if log_min_b is not None:
            log_min_boot.append(log_min_b)
            log_max_boot.append(log_max_b)

    return log_min_boot, log_max_boot


def _sigma_log(valores, metodo=None):
    """Incertidumbre de un límite en dex (log10 UA) a partir de sus réplicas bootstrap."""
    v = np.asarray(valores, dtype=float)
    v = v[np.isfinite(v)]
    if v.size < 5:
        return np.nan
    metodo = metodo or METODO_ERROR_REGION
    if metodo == "std":
        return _std_replicas(v)
    p16, p84 = np.percentile(v, [15.865, 84.135])
    return float((p84 - p16) / 2.0)


def _umbral_region(Z_pct, nivel_pico):
    """Umbral de densidad (%) que define la región estable según MODO_UMBRAL_REGION."""
    if MODO_UMBRAL_REGION != "hdr" or not np.size(Z_pct):
        return nivel_pico
    z = np.sort(np.asarray(Z_pct).ravel())[::-1]
    tot = z.sum()
    if tot <= 0:
        return nivel_pico
    i = min(int(np.searchsorted(np.cumsum(z) / tot, MASA_HDR)), z.size - 1)
    return float(z[i])


def _kde_cov_fija(x, y, w, cov, Xb, Yb):
    """KDE gaussiano ponderado con covarianza de kernel FIJA (sin normalizar)."""
    Hi = np.linalg.inv(np.asarray(cov, dtype=float))
    Z = np.zeros_like(Xb)
    for xi, yi, wi in zip(x, y, w):
        if wi <= 0:
            continue
        dx, dy = Xb - xi, Yb - yi
        Z += wi * np.exp(-0.5 * (Hi[0, 0] * dx * dx + 2.0 * Hi[0, 1] * dx * dy + Hi[1, 1] * dy * dy))
    return Z

In [ ]:
def _std_replicas(v):
    # Evita residuos de redondeo de np.std con réplicas idénticas.
    return 0.0 if np.all(v == v[0]) else float(np.std(v))


def _cobertura_gauss(k):
    """Cobertura de una gaussiana en ±kσ: 0.6827, 0.9545, 0.9973 para k = 1, 2, 3."""
    return math.erf(k / math.sqrt(2.0))


def _intervalo_sigma(valores_log, k, central=None, metodo=None):
    """
    Intervalo k-sigma (en log10 UA) de un límite a partir de sus réplicas bootstrap.
      "std"       : central ± k·std(réplicas)
      "percentil" : percentiles de las réplicas con la cobertura gaussiana de k·σ
    Devuelve (inf_log, sup_log) o None si hay menos de 5 réplicas válidas.
    """
    v = np.asarray(valores_log, dtype=float)
    v = v[np.isfinite(v)]
    if v.size < 5:
        return None
    metodo = metodo or METODO_SIGMA_DIBUJO
    if metodo == "std":
        c = float(np.mean(v)) if central is None else float(central)
        s = _std_replicas(v)
        if s == 0 and np.all(v == v[0]):
            return float(v[0]), float(v[0])
        return c - k * s, c + k * s
    cob = _cobertura_gauss(k)
    lo, hi = np.percentile(v, [50.0 * (1.0 - cob), 50.0 * (1.0 + cob)])
    return float(lo), float(hi)


def _dibujar_bandas_sigma(ax, valores_log, central, color, zorder, alpha_unico):
    """Bandas anidadas k-sigma (las más anchas primero). Con un solo k usa alpha_unico.
    Si DIBUJAR_BANDAS_SIGMA es False no dibuja nada (el bootstrap y los CSV no cambian)."""
    if not DIBUJAR_BANDAS_SIGMA:
        return
    ks = sorted(SIGMAS_DIBUJAR, reverse=True)
    alpha = alpha_unico if len(ks) == 1 else ALPHA_BANDA_SIGMA
    for k in ks:
        iv = _intervalo_sigma(valores_log, k, central)
        if iv is not None:
            ax.axvspan(iv[0], iv[1], color=color, alpha=alpha, zorder=zorder, linewidth=0)


def _filas_sigma(contexto: dict, boot: dict, ks=None):
    """Intervalos en formato largo con unidad explícita y NaN si faltan réplicas."""
    ks = SIGMAS if ks is None else ks
    filas = []
    for nombre, (central, vals) in boot.items():
        v = np.asarray(vals, dtype=float)
        v = v[np.isfinite(v)]
        valor = _desde_log10(central) if central is not None else np.nan
        mediana = _desde_log10(float(np.median(v))) if v.size else np.nan
        escala = valor * math.log(10.0)
        for metodo in ("std", "percentil"):
            for k in ks:
                iv = _intervalo_sigma(v, k, central, metodo)
                unidad = "UA" if nombre.startswith("a_") else "M_Tierra"
                es_ua = unidad == "UA"
                sigma = escala * _sigma_log(v, "std")
                robusta = escala * _sigma_log(v, "percentil")
                inf, sup = tuple(_desde_log10(val) for val in iv) if iv is not None else (np.nan, np.nan)
                filas.append({**contexto, "Limite": nombre, "Unidad": unidad, "Valor": valor,
                              "Sigma_std": sigma, "Sigma_robusta": robusta, "Mediana": mediana,
                              "Inf": inf, "Sup": sup, "Valor_UA": valor if es_ua else np.nan,
                              "Sigma_std_UA": sigma if es_ua else np.nan,
                              "Mediana_UA": mediana if es_ua else np.nan,
                              "Sigma_robusta_UA": robusta if es_ua else np.nan,
                              "Inf_UA": inf if es_ua else np.nan, "Sup_UA": sup if es_ua else np.nan,
                              "N_replicas": int(v.size), "error_estimado": iv is not None,
                              "Estado_error": "estimado" if iv is not None else "sin error estimado",
                              "Metodo": metodo, "k_sigma": k, "Cobertura_%": 100.0 * _cobertura_gauss(k)})
    return filas


def _guardar_tabla_sigma(filas, ruta):
    if not filas:
        return
    pd.DataFrame(filas).to_csv(ruta, index=False, encoding="utf-8-sig", na_rep="NaN")
    print(f"Intervalos σ guardados en: {ruta}")
    n_rep = min(f["N_replicas"] for f in filas)
    if max(SIGMAS) >= 3:
        cola = n_rep * (1 - _cobertura_gauss(3)) / 2
        if cola < 10:
            print(f"   Aviso: con {n_rep} réplicas cada cola de 3σ queda apoyada en ~{cola:.1f} "
                  f"réplicas; el 3σ por percentil es poco fiable. Sube N_BOOTSTRAP_INCERTIDUMBRE "
                  f"a 5000-10000 si lo vas a reportar.")


def _con_sufijo(nombre, sufijo):
    base, ext = os.path.splitext(nombre)
    return f"{base}{sufijo}{ext}"

Caché de KDE

In [ ]:
def _ruta_cache(base_dir):
    if not USAR_CACHE_KDE or base_dir is None:
        return None
    return os.path.join(os.path.abspath(base_dir), CACHE_KDE_NOMBRE)


def _set_cache_dir(base_dir):
    """Configura la caché de paneles llamados directamente; predecir_c pasa su ruta local."""
    global _CACHE_DIR
    _CACHE_DIR = _ruta_cache(base_dir)
    return _CACHE_DIR


def _hash_cache(*partes):
    """Incluye tipo y longitud para distinguir arrays y concatenaciones ambiguas."""
    h = hashlib.sha256(_CACHE_VERSION.encode())
    for p in partes:
        if isinstance(p, np.ndarray):
            dato = repr((p.shape, p.dtype.str)).encode() + b"\0" + np.ascontiguousarray(p).tobytes()
        else:
            dato = repr(p).encode()
        h.update(len(dato).to_bytes(8, "little"))
        h.update(dato)
    return h.hexdigest()[:24]


def _cache_obtener(etiqueta, clave, calcular, cache_dir=None):
    """Carga una caché local de confianza y publica cada escritura por reemplazo atómico."""
    if not USAR_CACHE_KDE or cache_dir is None:
        return calcular()
    nombre = re.sub(r"[^\w\-]+", "_", str(etiqueta)).strip("_")[:60]
    ruta = os.path.join(cache_dir, f"kde_{nombre}_{clave}.pkl")
    if not FORZAR_RECALCULO_KDE:
        try:
            with open(ruta, "rb") as f:
                res = pickle.load(f)
            if not isinstance(res, dict) or not {"malla", "regiones", "boot"} <= res.keys():
                raise ValueError("formato KDE inesperado")
        except FileNotFoundError:
            pass
        except (OSError, EOFError, pickle.UnpicklingError, ImportError, AttributeError, ValueError) as exc:
            print(f"   Aviso: caché ilegible ({exc}); se recalcula.")
        else:
            print(f"   KDE cargado desde caché: {os.path.basename(ruta)}")
            return res
    res = calcular()
    temporal = None
    try:
        os.makedirs(cache_dir, exist_ok=True)
        with tempfile.NamedTemporaryFile(dir=cache_dir, suffix=".tmp", delete=False) as f:
            temporal = f.name
            pickle.dump(res, f, protocol=pickle.HIGHEST_PROTOCOL)
        os.replace(temporal, ruta)
    except (OSError, pickle.PicklingError) as exc:
        print(f"   Aviso: no se pudo guardar la caché ({exc}).")
    finally:
        if temporal and os.path.exists(temporal):
            try:
                os.unlink(temporal)
            except OSError as exc:
                print(f"   Aviso: no se pudo quitar el archivo temporal de caché ({exc}).")
    return res

Leyenda

In [ ]:
def _cfg_leyenda(tipo=None, extra=None):
    """Configuración de leyenda: defecto + ajuste del tipo espectral + extra."""
    cfg = dict(LEYENDA_DEFECTO)
    cfg.update(LEYENDA_POR_TIPO.get(tipo, {}))
    cfg.update(extra or {})
    return cfg


def _n_entradas_leyenda(d, lineas_extra=None):
    """Estimación conservadora: sistemas + Tierra y Júpiter + hasta 3 regiones
    (a_min/a_max y m_min/m_max = 12 entradas)."""
    return d["Sistema"].nunique() + 2 + 12 + len(lineas_extra or [])


def _filas_leyenda(n_ent, cfg):
    ncol = cfg.get("ncol") or (2 if n_ent <= 8 else 3)
    return math.ceil(n_ent / ncol)


def _alto_leyenda(filas, cfg):
    """Altura (pulgadas) que ocupa una leyenda de `filas` filas con la fuente de cfg."""
    return 0.2 * filas * cfg["fs"] / 6.3

Parámetros de los paneles compactos

In [ ]:
JUPITER_UA, JUPITER_MASA = 5.2, 317.8
COLOR_TIERRA = "#1e90ff"
COLOR_JUPITER = "#e08a3c"

# Región más estable (líneas de región a_min / a_max): densidad relativa >= 90.9 %
NIVEL_REGION_ESTABLE = 90.9

# Zoom por tipo espectral en plot_figuras_por_tipo: {tipo: (margen_x, margen_y)} en dex.
# Para quitar el zoom: ZOOM_POR_TIPO = {}
ZOOM_POR_TIPO = {"F": (0.6, 0.8)}

# ISOLÍNEAS DIFUMINADAS: se desvanecen (menor opacidad y grosor) lejos del centro.
NIVELES_REGIONES = (90.9, 80.8, 70.7, 60.6, 50.5, 40.4, 30.3, 20.2, 10.1)
ALPHA_CENTRO, ALPHA_BORDE = 0.95, 0.15
LW_CENTRO, LW_BORDE = 1.6, 0.6


def _dibujar_isolineas_difuminadas(ax, X, Y, Z_pct, niveles=NIVELES_REGIONES,
                                   color="white"):
    """
    Dibuja una isolínea por nivel de densidad relativa (%). El nivel más alto
    (centro) es el más nítido y los niveles bajos se van desvaneciendo.
    """
    n = len(niveles)
    zmin, zmax = float(Z_pct.min()), float(Z_pct.max())
    for k, nv in enumerate(sorted(niveles, reverse=True)):
        if not (zmin < nv < zmax):
            continue
        t = k / (n - 1) if n > 1 else 0.0
        alpha = ALPHA_CENTRO + (ALPHA_BORDE - ALPHA_CENTRO) * t
        lw = LW_CENTRO + (LW_BORDE - LW_CENTRO) * t
        ax.contour(X, Y, Z_pct, levels=[nv], colors=[color],
                   linewidths=lw, alpha=alpha, zorder=4)


def _limites_por_nivel(Z_pct: np.ndarray, X: np.ndarray, nivel: float):
    """Mínimo y máximo de log(distancia) donde la densidad relativa >= nivel (%)."""
    xs = X[Z_pct >= nivel]
    if xs.size == 0:
        return None, None
    return float(xs.min()), float(xs.max())


def _regiones_por_umbral(Z_pct, X, umbral, gap_dex=None, Y=None, detalles=False):
    """Rangos en distancia por umbral; conserva el agrupamiento histórico por gap."""
    gap_dex = GAP_REGIONES_DEX if gap_dex is None else gap_dex
    if Y is None:
        Y = np.broadcast_to(np.arange(Z_pct.shape[1], dtype=float)[None, :], Z_pct.shape)
    mascara = np.isfinite(Z_pct) & (Z_pct >= umbral)
    cols = np.flatnonzero(mascara.any(axis=1))
    if not cols.size:
        return []
    cortes = np.flatnonzero(np.diff(X[cols, 0]) > gap_dex + 1e-12) + 1
    regiones = []
    for tramo in np.split(cols, cortes):
        comp = np.zeros_like(mascara)
        comp[tramo] = mascara[tramo]
        pico = np.unravel_index(np.argmax(np.where(comp, Z_pct, -np.inf)), Z_pct.shape)
        t, d = _geometria_isla(Z_pct, X, Y, comp, umbral, pico)
        regiones.append((t, d) if detalles else t[:2])
    return regiones


def _fusionar_islas(tramos, gap_dex):
    """
    Fusiona islas SOLO si están cerca en distancia (x) Y en masa (y).
    Cada tramo: [lo, hi, ylo, yhi, ypico, zpico]. Dos islas con el mismo rango de distancia pero
    en masas distintas (p. ej. F: isla principal e isla cercana a gigantes gaseosos) NO se fusionan.
    """
    fusion = [list(t) for t in tramos]
    cambio = True
    while cambio:
        cambio = False
        for i, j in itertools.combinations(range(len(fusion)), 2):
            a, b = fusion[i], fusion[j]
            gx = max(a[0], b[0]) - min(a[1], b[1])      # < 0 si se solapan en x
            gy = max(a[2], b[2]) - min(a[3], b[3])      # < 0 si se solapan en y
            if gx <= gap_dex and gy <= gap_dex:
                fusion[i] = [min(a[0], b[0]), max(a[1], b[1]),
                             min(a[2], b[2]), max(a[3], b[3]),
                             a[4] if a[5] >= b[5] else b[4], max(a[5], b[5])]
                del fusion[j]
                cambio = True
                break
    return fusion


def _nivel_isla(h):
    """Mayor nivel de la barra estrictamente inferior al pico."""
    return max((float(nv) for nv in NIVELES_REGIONES if nv < h), default=None)


def _geometria_isla(Z, X, Y, comp, nivel, pico):
    """Extremos del contorno interpolado por aristas en la componente del pico."""
    xm, ym = X[comp], Y[comp]
    nodos = (float(xm.min()), float(xm.max()), float(ym.min()), float(ym.max()))
    xs, ys = list(nodos[:2]), list(nodos[2:])
    # Los extremos x/y del contorno bilineal se alcanzan en aristas de celdas.
    for axis, coord, bordes in ((0, X, xs), (1, Y, ys)):
        a, b = np.swapaxes(Z, 0, axis), np.swapaxes(coord, 0, axis)
        c = np.swapaxes(comp, 0, axis)
        cruce = (c[:-1] & (a[1:] < nivel)) | (c[1:] & (a[:-1] < nivel))
        if not cruce.any():
            continue
        f = (nivel - a[:-1][cruce]) / (a[1:][cruce] - a[:-1][cruce])
        v = b[:-1][cruce] + f * (b[1:][cruce] - b[:-1][cruce])
        bordes.extend((v.min(), v.max()))
    exactos = (float(min(xs)), float(max(xs)), float(min(ys)), float(max(ys)))
    return (*exactos, float(Y[pico])), {"nodos": nodos, "nivel": nivel, "comp": comp,
                                       "pico": (float(X[pico]), float(Y[pico]))}


def _regiones_por_picos(Z_pct, X, Y, nivel_local=None, nivel_min=None, gap_dex=None, n_max=None, detalles=False):
    """
    Componentes conexas de cuatro vecinos con umbral inferior al pico.
    Eleva el nivel cuando comparte componente con un pico mayor.
    gap_dex se conserva en la firma; cada isla conserva su identidad.
    """
    nivel_local = NIVEL_REGION_ESTABLE if nivel_local is None else nivel_local
    nivel_min = NIVEL_MIN_PICO if nivel_min is None else nivel_min
    if (Z_pct.ndim != 2 or min(Z_pct.shape) < 2 or not np.isfinite(Z_pct).all()
            or Z_pct.max() <= 0 or n_max == 0):
        return []
    # Ventanas en dex por eje, con un número impar de nodos centrados.
    pasos = (X[1, 0] - X[0, 0], Y[0, 1] - Y[0, 0])
    if min(pasos) <= 0:
        raise ValueError("La malla debe ser creciente.")
    ventana = tuple(max(3, 2 * min(int(round(PICO_VENTANA_DEX / (2 * p))), n) + 1)
                    for p, n in zip(pasos, Z_pct.shape))
    es_pico = (Z_pct == maximum_filter(Z_pct, size=ventana, mode="nearest")) & (Z_pct >= nivel_min)
    ij = np.argwhere(es_pico)
    ij = ij[np.argsort(-Z_pct[ij[:, 0], ij[:, 1]], kind="stable")]
    picos_previos, tramos = [], []
    cubiertos = np.zeros_like(es_pico)
    etiquetas = {}

    def componente(t, pico):
        if t not in etiquetas:
            etiquetas[t] = _label_cc(Z_pct >= t)[0]
        lab = etiquetas[t]
        return lab == lab[pico]

    for i, j in ij:
        if n_max is not None and len(tramos) >= n_max:
            break
        if cubiertos[i, j]:
            continue
        pico = (i, j)
        h = float(Z_pct[pico])
        t = _nivel_isla(h) if NIVELES_POR_ALTURA else h * nivel_local / 100.0
        if t is None or t > h:
            continue
        comp = componente(t, pico)
        while t < h and any(comp[p] for p in picos_previos):
            t = min(t + 0.25, h)
            comp = componente(t, pico)
        if any(comp[p] for p in picos_previos):
            continue
        cubiertos |= comp
        picos_previos.append(pico)
        tramos.append(_geometria_isla(Z_pct, X, Y, comp, t, pico))
    tramos.sort(key=lambda tramo: (tramo[0][0], tramo[0][2]))
    return tramos if detalles else [t for t, _ in tramos]


def _error_asimetrico(valor_ua, replicas_log):
    """(dn, up) en UA (o M_Tierra si es una masa) a partir de las réplicas bootstrap (log10); ambos >= 0.
    Con METODO_ERROR_REGION = "percentil" los percentiles 15.9 / 84.1 se miden respecto a la MEDIANA de
    las réplicas (antes respecto al valor nominal: si este caía fuera del intervalo, un lado daba 0)."""
    v = np.asarray(replicas_log, dtype=float)
    v = v[np.isfinite(v)]
    if v.size < 5 or not np.isfinite(valor_ua) or valor_ua <= 0:
        return np.nan, np.nan
    if np.all(v == v[0]):
        return 0.0, 0.0
    if METODO_ERROR_REGION == "std":
        la = np.log10(valor_ua)
        sg = _std_replicas(v)
        lo, hi = la - sg, la + sg
        return valor_ua - _desde_log10(lo), _desde_log10(hi) - valor_ua
    lo, med, hi = np.percentile(v, [15.865, 50.0, 84.135])
    bajo, centro, alto = (_desde_log10(x) for x in (lo, med, hi))
    return centro - bajo, alto - centro


def _txt_lim(nombre, a, dn, up):
    """Etiqueta de distancia con precisión que no oculta errores pequeños."""
    valor = f"$a_{{{nombre}}}$ = {a:.5g} AU"
    if not np.isfinite([dn, up]).all():
        return valor + " (sin error estimado)"
    if ERROR_ASIMETRICO:
        error = f"{a:.5g}^{{+{up:.2g}}}_{{-{dn:.2g}}}"
        return f"$a_{{{nombre}}}$ = $" + error + "$ AU"
    return valor + f" $\\pm$ {(dn + up) / 2:.2g} AU"

Límites en masa de cada región

In [ ]:
def _limites_masa_region(Z_pct, X, Y, lo, hi):
    """
    (Solo para el modo "umbral" o tramos sin información de masa.)
    Extensión en log10(masa) de la isla de densidad cuya distancia está en [lo, hi].
    Mismo criterio que las regiones en distancia: Z >= NIVEL_REGION_ESTABLE % del pico
    de la isla ("picos") o del máximo global ("umbral").
    Devuelve (ymin, ymax, y_del_pico) o None.
    En modo "picos" los límites de masa vienen directamente de la componente conexa de cada isla.
    """
    banda = (X >= lo - 1e-9) & (X <= hi + 1e-9)
    if not banda.any():
        return None
    zb = np.where(banda, Z_pct, -np.inf)
    pico = float(zb.max())
    ref = pico if MODO_REGIONES == "picos" else 100.0
    mask = banda & (Z_pct >= NIVEL_REGION_ESTABLE / 100.0 * ref)
    if not mask.any():
        return None
    ys = Y[mask]
    y_pico = float(Y[np.unravel_index(np.argmax(zb), zb.shape)])
    return float(ys.min()), float(ys.max()), y_pico


def _masa_region_nominal(malla, lmin, lmax):
    """(log m_min, log m_max) de la región con distancia en [lmin, lmax], desde la malla guardada.
    (Ya no se usa al dibujar: las masas de cada isla vienen en el resultado de calcular_kde_panel.)"""
    if malla is None:
        return None
    X, Y, Z = malla
    m = _limites_masa_region(Z, X, Y, lmin, lmax)
    return None if m is None else (m[0], m[1])


def _txt_masa(nombre, m, dn=None, up=None):
    """Etiqueta de un límite de masa: m (+up/-dn) M_Tierra (sin error si no hay réplicas)."""
    if dn is None or up is None or not np.isfinite([dn, up]).all():
        return f"$m_{{{nombre}}}$ = {m:.4g} $M_\\oplus$ (sin error estimado)"
    return f"$m_{{{nombre}}}$ = ${m:.4g}^{{+{up:.2g}}}_{{-{dn:.2g}}}$ $M_\\oplus$"


def _region_principal(regs, malla):
    """
    Región que contiene el máximo global de densidad (la "principal"). Si ninguna lo contiene,
    la más cercana en distancia. Las cajas sobre el panel muestran a_min / a_max de ESTA región,
    no el a_min de la primera y el a_max de la última (que mezclaba regiones distintas).
    """
    if not regs:
        return None
    if malla is None:
        return regs[0]
    X, Y, Z = malla
    i, j = np.unravel_index(np.argmax(Z), Z.shape)
    px, py = float(X[i, j]), float(Y[i, j])
    for r in regs:
        if r.get("comp") is not None:
            if r["comp"][i, j]:
                return r
            continue
        if r["lmin"] - 1e-9 <= px <= r["lmax"] + 1e-9:
            if "log_m_min" not in r or r["log_m_min"] - 1e-9 <= py <= r["log_m_max"] + 1e-9:
                return r
    return min(regs, key=lambda r: abs((r["lmin"] + r["lmax"]) / 2.0 - px))


def _centro_tramo(t):
    """Centro (log a, log m) de un tramo; log m = None si el tramo solo trae distancia (modo umbral)."""
    cx = (t[0] + t[1]) / 2.0
    cy = (t[2] + t[3]) / 2.0 if len(t) >= 5 else None
    return cx, cy


def _dist_norm(c, c0):
    """Distancia normalizada entre dos centros: 1.0 = borde de la elipse de tolerancia."""
    d = abs(c[0] - c0[0]) / TOL_ASIGNACION_DEX
    if c[1] is not None and c0[1] is not None:
        d = math.hypot(d, (c[1] - c0[1]) / TOL_ASIGNACION_Y_DEX)
    return d


def _bootstrap_regiones(x, y, weights, xlims, ylims, umbral, regiones_nom,
                        n_bootstrap=None, grid_res=None, rng=None, bw=KDE_BW,
                        grupos=None, cov=None, buscador=None):
    """
    Bootstrap de los bordes de CADA región nominal (distancia y masa). En cada réplica se recalculan
    los tramos y cada uno se asigna a la región nominal más cercana EN EL PLANO (log a, log m);
    la asignación es uno a uno y se descartan pares fuera de la elipse de tolerancia. Devuelve una lista, una entrada por región nominal:
    {"min": [...], "max": [...], "ymin": [...], "ymax": [...]}  (log10;
    "min"/"max" en UA y "ymin"/"ymax" en M_Tierra).
      grupos : etiqueta de sistema de cada planeta -> se remuestrean SISTEMAS completos.
      buscador : callable(Z_pct, X, Y) -> [tramos]; por defecto el umbral global.
      cov    : covarianza del kernel del ajuste nominal -> ancho de banda fijo en todas las
               réplicas (si KDE_COV_FIJA). Sin ella se usa gaussian_kde(bw_method=bw).
    """
    n_bootstrap = _n_replicas(n_bootstrap, N_BOOTSTRAP_INCERTIDUMBRE)
    grid_res = grid_res or BOOTSTRAP_GRID_RES
    if rng is None:
        rng = np.random.default_rng()
    n = len(x)
    out = [{"min": [], "max": [], "ymin": [], "ymax": []} for _ in regiones_nom]
    centros = [_centro_tramo(t) for t in regiones_nom]

    if n == 0 or not centros or n_bootstrap == 0:
        return out
    Xb, Yb = np.mgrid[xlims[0]:xlims[1]:grid_res, ylims[0]:ylims[1]:grid_res]
    pos_b = np.vstack([Xb.ravel(), Yb.ravel()])
    usar_cov = bool(KDE_COV_FIJA and cov is not None)

    miembros = None
    if grupos is not None:
        _, inv = np.unique(np.asarray(grupos).astype(str), return_inverse=True)
        miembros = [np.where(inv == g)[0] for g in range(inv.max() + 1)]
        if len(miembros) < 2:
            return out  # Un solo sistema no permite estimar varianza entre sistemas.

    for _ in tqdm(range(n_bootstrap), desc="   Bootstrap regiones", leave=False, unit="rep"):
        if miembros is not None:
            sel = rng.integers(0, len(miembros), size=len(miembros))
            idx = np.concatenate([miembros[s] for s in sel])
        else:
            idx = rng.integers(0, n, size=n)
        x_b, y_b, w_b = x[idx], y[idx], weights[idx]
        if not _pesos_validos(w_b):
            continue
        if usar_cov:
            # Una réplica de un solo sistema sigue siendo válida con un kernel fijo.
            mult = np.bincount(idx, minlength=n)
            Z_b = _kde_cov_fija(x, y, weights * mult, cov, Xb, Yb)
        else:
            kde_b = _ajustar_kde(x_b, y_b, w_b, bw=bw)
            if kde_b is None:
                continue
            Z_b = kde_b(pos_b).reshape(Xb.shape)
        Z_b = _normalizar_campo(Z_b)
        if Z_b is None:
            continue

        if buscador is not None:
            tramos = buscador(Z_b, Xb, Yb)
        else:
            tramos = _regiones_por_umbral(Z_b, Xb, _umbral_region(Z_b, umbral))
        dist = np.array([[_dist_norm(_centro_tramo(t), c0) for c0 in centros] for t in tramos])
        if not dist.size:
            continue
        # Una sola isla por región y réplica: fusionarlas inflaba los bordes.
        coste = np.where(dist <= 1.0, dist, dist.size + 1.0)
        indices, regiones = linear_sum_assignment(coste)
        for j, k in zip(indices, regiones):
            if dist[j, k] > 1.0:
                continue
            t = tramos[j]
            lo, hi = t[:2]
            ylo, yhi = t[2:4] if len(t) >= 5 else (None, None)
            out[k]["min"].append(lo)
            out[k]["max"].append(hi)
            if ylo is None:           # modo umbral: masa por banda en x
                m = _limites_masa_region(Z_b, Xb, Yb, lo, hi)
                if m is None:
                    continue
                ylo, yhi = m[0], m[1]
            out[k]["ymin"].append(ylo)
            out[k]["ymax"].append(yhi)
    return out

Cálculo (KDE + regiones + bootstrap) y dibujo de panel

In [ ]:
# ===========================================================================
LIMITES_REGION = ("a_min", "a_max", "m_min", "m_max")
_CACHE_HEREDADA = object()


def _registrar_region(res, tramo, geometria, fuentes):
    """Registra los mismos bordes y errores para KDE de referencia y de un punto."""
    k = len(res["regiones"]) + 1
    lmin, lmax, ylo, yhi, ypico = tramo
    r = {"region": k, "lmin": lmin, "lmax": lmax, "log_m_min": ylo, "log_m_max": yhi,
         "m_pico": _desde_log10(ypico), **geometria}
    X, Y, _ = res["malla"]
    r.update(m_min_en_borde=bool(ylo <= Y[0, 0]), m_max_en_borde=bool(yhi >= Y[0, -1]),
             a_min_en_borde=bool(lmin <= X[0, 0]), a_max_en_borde=bool(lmax >= X[-1, 0]))
    r["fuentes"] = fuentes
    errores = (("dn_min", "up_min"), ("dn_max", "up_max"),
               ("dnm_min", "upm_min"), ("dnm_max", "upm_max"))
    for nombre, central, (dn, up) in zip(LIMITES_REGION, tramo[:4], errores):
        replicas = fuentes["total"].get(nombre, [])
        res["boot"][f"{nombre}_{k}"] = (central, replicas)
        r[nombre] = _desde_log10(central)
        if not np.isfinite(r[nombre]):
            res.setdefault("avisos", []).append(f"Región {k}: {nombre} fuera del rango numérico lineal; se conserva en dex.")
        r[dn], r[up] = _error_asimetrico(r[nombre], replicas)
    r["s_min"] = (r["dn_min"] + r["up_min"]) / 2.0
    r["s_max"] = (r["dn_max"] + r["up_max"]) / 2.0
    res["regiones"].append(r)


# CAPA 1 — CÁLCULO (KDE + regiones + bootstrap), con caché en disco
#          No dibuja nada: devuelve un dict serializable.
# ===========================================================================
def calcular_kde_panel(df_subset, xlims, ylims, modo_peso=None, n_bootstrap=None,
                       nivel_min_pico=None, etiqueta="panel", n_regiones=None, *, cache_dir=_CACHE_HEREDADA):
    """KDE 2D, regiones y bootstrap. Devuelve malla, regiones y réplicas por borde."""
    _validar_malla(xlims, ylims)
    modo = modo_peso or MODO_PESO_DEFECTO
    nb = _n_replicas(n_bootstrap, N_BOOTSTRAP_INCERTIDUMBRE)
    nmp = NIVEL_MIN_PICO if nivel_min_pico is None else float(nivel_min_pico)
    if not 0 <= nmp <= 100:
        raise ValueError("nivel_min_pico debe estar entre 0 y 100.")
    if n_regiones is not None:
        n_regiones = _n_replicas(n_regiones, 0)
    df_valid = _datos_kde(df_subset)
    w = _calcular_pesos(df_valid, modo)
    positivos = np.isfinite(w) & (w > 0)
    x = np.log10(df_valid.loc[positivos, "Distancia_UA"].to_numpy())
    y = np.log10(df_valid.loc[positivos, "Masa_Tierra"].to_numpy())
    weights = w[positivos]
    grupos = (df_valid.loc[positivos, "Sistema"].to_numpy(dtype=str)
              if BOOT_POR_SISTEMAS and "Sistema" in df_valid else None)
    problemas = len(df_subset) - len(x)
    avisos = [f"{problemas} fila(s) sin coordenadas válidas o peso positivo para el KDE."] if problemas else []
    if cache_dir is _CACHE_HEREDADA:
        cache_dir = _CACHE_DIR
    clave = _hash_cache(
        x, y, weights, tuple(xlims), tuple(ylims), modo, nb, nmp, MODO_REGIONES,
        NIVEL_REGION_ESTABLE, PICO_VENTANA_DEX, GAP_REGIONES_DEX, TOL_ASIGNACION_DEX,
        TOL_ASIGNACION_Y_DEX, MODO_UMBRAL_REGION, MASA_HDR, KDE_COV_FIJA, KDE_BW,
        KDE_GRID_RES, BOOT_POR_SISTEMAS, METODO_ERROR_REGION, BOOTSTRAP_GRID_RES,
        SEMILLA_BOOTSTRAP, NIVELES_POR_ALTURA, tuple(NIVELES_REGIONES), n_regiones,
        grupos if grupos is not None else "sin_grupos", problemas)

    def _calc():
        res = {"malla": None, "regiones": [], "boot": {}, "avisos": avisos.copy()}
        kde = _ajustar_kde(x, y, weights, bw=KDE_BW)
        if kde is None:
            res["avisos"].append("KDE de referencia no disponible: se requieren al menos tres puntos "
                                 "con peso positivo y rango dos en log(a), log(m).")
            return res
        X, Y = np.mgrid[xlims[0]:xlims[1]:KDE_GRID_RES, ylims[0]:ylims[1]:KDE_GRID_RES]
        Z_pct = _normalizar_campo(kde(np.vstack([X.ravel(), Y.ravel()])).reshape(X.shape))
        if Z_pct is None:
            res["avisos"].append("Campo KDE no representable en la malla solicitada.")
            return res
        res["malla"] = (X, Y, Z_pct)

        def buscar(Zg, Xg, Yg, detalles=False):
            if MODO_REGIONES == "picos":
                return _regiones_por_picos(Zg, Xg, Yg, nivel_min=nmp, n_max=n_regiones, detalles=detalles)
            if MODO_REGIONES == "umbral":
                umbral = _umbral_region(Zg, NIVEL_REGION_ESTABLE)
                regiones = _regiones_por_umbral(Zg, Xg, umbral, Y=Yg, detalles=True)
                return regiones if detalles else [t for t, _ in regiones]
            raise ValueError(f"MODO_REGIONES desconocido: {MODO_REGIONES!r}")

        regs = buscar(Z_pct, X, Y, detalles=True)
        if not regs:
            res["avisos"].append("Ninguna isla cumple el criterio de detección.")
            return res
        if grupos is not None and len(np.unique(grupos)) < 2:
            res["avisos"].append("Un solo sistema de referencia: bootstrap entre sistemas no estimable.")
        boot_reg = _bootstrap_regiones(
            x, y, weights, xlims, ylims, NIVEL_REGION_ESTABLE, [t for t, _ in regs],
            n_bootstrap=nb, rng=np.random.default_rng(SEMILLA_BOOTSTRAP), bw=KDE_BW,
            grupos=grupos, cov=kde.covariance, buscador=buscar)
        fuente = "bootstrap_sistemas" if grupos is not None else "bootstrap_filas"
        res["fuente_incertidumbre"] = fuente
        for (tramo, geometria), rep in zip(regs, boot_reg):
            replicas = dict(zip(LIMITES_REGION, (rep["min"], rep["max"], rep["ymin"], rep["ymax"])))
            _registrar_region(res, tramo, geometria, {"total": replicas, fuente: replicas})
        return res

    # Una semilla aleatoria pide una nueva estimación cada vez.
    return _cache_obtener(etiqueta, clave, _calc, cache_dir if SEMILLA_BOOTSTRAP is not None else None)


# CAPA 2 — DISEÑO (solo dibuja a partir del resultado de la capa 1)


def dibujar_panel_kde(ax, df_subset, res, titulo, estilos_sistema, xlims, ylims,
                      cmap="magma", mostrar_ylabel=True, lineas_extra=None, leyenda=None):
    """
    Dibuja un panel compacto con los resultados `res` de calcular_kde_panel:
    mapa de densidad + isolíneas difuminadas, líneas de a_min / a_max por región (verticales, de altura
    completa), RECTÁNGULO de masas de cada región (líneas horizontales m_min / m_max entre a_min y
    a_max + relleno tenue), cajas con los valores de la región principal, puntos por sistema,
    Tierra y Júpiter y una leyenda debajo.
    `leyenda` es un dict como LEYENDA_DEFECTO (fs, ms, ncol, y, fs_cajas).
    """
    cfg = {**LEYENDA_DEFECTO, **(leyenda or {})}
    ax.set_facecolor("black")
    df_valid = _datos_kde(df_subset)

    # ---------------- Mapa de densidad ----------------
    if res["malla"] is not None:
        X, Y, Z_pct = res["malla"]
        ax.contourf(X, Y, Z_pct, levels=np.linspace(0.0, 100.0, 100),
                    cmap=cmap, extend="min", zorder=1)
        if DIBUJAR_ISOLINEAS:
            _dibujar_isolineas_difuminadas(ax, X, Y, Z_pct)

    # ---------------- Regiones (líneas a_min / a_max y m_min / m_max) ----------------
    regs, boot = res["regiones"], res["boot"]
    multi = len(regs) > 1
    handles_lim = []
    handles_masa = []
    for r in regs:
        k = r["region"]
        ls_min, ls_max = LS_REGIONES[(k - 1) % len(LS_REGIONES)]
        col_banda = COLORES_BANDAS_REGION[(k - 1) % len(COLORES_BANDAS_REGION)]
        col_reg = COLORES_REGIONES[(k - 1) % len(COLORES_REGIONES)]
        for lval, nom, ls in ((r["lmin"], "a_min", ls_min), (r["lmax"], "a_max", ls_max)):
            _dibujar_bandas_sigma(ax, boot[f"{nom}_{k}"][1], lval, col_banda, 3, 0.15)
            ax.axvline(lval, color=col_reg, linestyle=ls, linewidth=1.6, zorder=6)

        # límites en masa (líneas horizontales) de la propia isla: cierran el rectángulo de la región
        mm = ((r["log_m_min"], r["log_m_max"]) if DIBUJAR_LIMITES_MASA and "log_m_min" in r else None)
        if mm is not None:
            if RELLENO_CAJA_MASA > 0 and not LIMITES_MASA_ANCHO_COMPLETO:
                ax.add_patch(Rectangle((r["lmin"], mm[0]), r["lmax"] - r["lmin"], mm[1] - mm[0],
                                       facecolor=col_reg, edgecolor="none",
                                       alpha=RELLENO_CAJA_MASA, zorder=3))
            for yv, ls in ((mm[0], ls_min), (mm[1], ls_max)):
                if LIMITES_MASA_ANCHO_COMPLETO:
                    ax.axhline(yv, color=col_reg, linestyle=ls, linewidth=1.6, zorder=6)
                else:
                    ax.hlines(yv, r["lmin"], r["lmax"], colors=col_reg, linestyles=ls,
                              linewidth=1.8, zorder=6)
            s_min = f"min,{k}" if multi else "min"
            s_max = f"max,{k}" if multi else "max"
            handles_masa += [
                Line2D([], [], color=col_reg, ls=ls_min, lw=1.6,
                       label=_txt_masa(s_min, r["m_min"], r.get("dnm_min"), r.get("upm_min"))),
                Line2D([], [], color=col_reg, ls=ls_max, lw=1.6,
                       label=_txt_masa(s_max, r["m_max"], r.get("dnm_max"), r.get("upm_max"))),
            ]

        sub_min = f"min,{k}" if multi else "min"
        sub_max = f"max,{k}" if multi else "max"
        handles_lim += [
            Line2D([], [], color=col_reg, ls=ls_min, lw=1.6,
                   label=_txt_lim(sub_min, r["a_min"], r["dn_min"], r["up_min"])),
            Line2D([], [], color=col_reg, ls=ls_max, lw=1.6,
                   label=_txt_lim(sub_max, r["a_max"], r["dn_max"], r["up_max"])),
        ]

    if regs:
        # Cajas sobre el panel: a_min y a_max de la región PRINCIPAL (la que contiene el máximo
        # global de densidad). Las demás regiones están en la leyenda.
        caja = dict(facecolor="white", alpha=0.95, edgecolor="black",
                    boxstyle="square,pad=0.2", linewidth=0.6)
        rp = _region_principal(regs, res["malla"])
        ax.text(0.0, 1.015, _txt_lim("min", rp["a_min"], rp["dn_min"], rp["up_min"]),
                transform=ax.transAxes, ha="left", va="bottom", fontsize=cfg["fs_cajas"],
                fontweight="bold", bbox=caja)
        ax.text(1.0, 1.015, _txt_lim("max", rp["a_max"], rp["dn_max"], rp["up_max"]),
                transform=ax.transAxes, ha="right", va="bottom", fontsize=cfg["fs_cajas"],
                fontweight="bold", bbox=caja)

    # ---------------- Líneas verticales extra opcionales ----------------
    handles_extra = []
    for etiqueta, ua, color, ls in (lineas_extra or []):
        ax.axvline(np.log10(ua), color=color, linestyle=ls, linewidth=1.4, zorder=6)
        handles_extra.append(Line2D([], [], color=color, ls=ls, lw=1.4,
                                    label=f"{etiqueta} = {ua:.5g} AU"))

    # ---------------- Puntos por sistema ----------------
    nombre_ss = SISTEMA_SOLAR["nombre_sistema"]
    handles_sis = []
    for sist in df_valid["Sistema"].unique():
        d = df_valid[df_valid["Sistema"] == sist]
        p = estilos_sistema.get(sist, {"marker": "o", "color": "white"})
        es_ss = (sist == nombre_ss)
        h = ax.scatter(np.log10(d["Distancia_UA"]), np.log10(d["Masa_Tierra"]),
                       marker=p["marker"], color=p["color"], s=95 if es_ss else 38,
                       edgecolor="black" if es_ss else "white",
                       linewidth=0.7 if es_ss else 0.5, label=sist,
                       zorder=10 if es_ss else 5)
        handles_sis.append(h)

    # ---------------- Tierra y Júpiter ----------------
    ax.scatter(0, 0, s=150, color=COLOR_TIERRA, edgecolor="white", linewidth=0.8, zorder=9)
    ax.text(0, -0.38, "Earth", color=COLOR_TIERRA, fontsize=8, fontweight="bold",
            ha="center", va="top", zorder=9, clip_on=True)
    xj, yj = np.log10(JUPITER_UA), np.log10(JUPITER_MASA)
    ax.scatter(xj, yj, s=150, color=COLOR_JUPITER, edgecolor="white", linewidth=0.8, zorder=9)
    ax.text(xj, yj - 0.38, "Jupiter", color=COLOR_JUPITER, fontsize=8, fontweight="bold",
            ha="center", va="top", zorder=9, clip_on=True)
    handles_ref = [
        Line2D([], [], marker="o", ls="", color=COLOR_TIERRA, markeredgecolor="white",
               markersize=cfg["ms"], label=r"Earth (1 AU, 1 $M_\oplus$)"),
        Line2D([], [], marker="o", ls="", color=COLOR_JUPITER, markeredgecolor="white",
               markersize=cfg["ms"], label=r"Jupiter (5.2 AU, 317.8 $M_\oplus$)"),
    ]

    # ---------------- Formato ----------------
    ax.set_title(titulo, fontsize=12, fontweight="bold", pad=24)
    ax.set_xlim(xlims)
    ax.set_ylim(ylims)
    ax.set_box_aspect(1)
    ax.set_anchor("N")                       # alinear arriba; la leyenda cae debajo
    ax.set_xlabel("log[Distance] (AU)", fontsize=10)
    if mostrar_ylabel:
        ax.set_ylabel(r"log[Planetary Mass] ($M_\oplus$)", fontsize=10)
    ax.tick_params(labelsize=8)
    ax.grid(True, ls="--", color="white", alpha=0.15, zorder=2)

    # ---------------- Leyenda (tamaño y posición desde cfg) ----------------
    handles = handles_sis + handles_ref + handles_lim + handles_masa + handles_extra
    n = len(handles)
    ncol = cfg.get("ncol") or (2 if n <= 8 else 3)
    esc = cfg["fs"] / 6.3                     # los espaciados escalan con la fuente
    ax.legend(handles=handles, loc="upper center", bbox_to_anchor=(0.5, cfg["y"]),
              ncol=ncol, fontsize=cfg["fs"], frameon=True, framealpha=0.95,
              handletextpad=0.3 * esc, columnspacing=0.8 * esc, borderpad=0.4 * esc,
              labelspacing=0.25 * esc, markerscale=max(1.0, 0.8 * esc))

    return {"regiones": regs, "boot": boot, "malla": res["malla"], "n_leyenda": n}


def _dibujar_panel_matriz(ax, df_subset, titulo, estilos_sistema, xlims, ylims,
                          cmap="magma", mostrar_ylabel=True, lineas_extra=None,
                          color_limites=COLOR_LIMITES, modo_peso=None, n_bootstrap=None,
                          nivel_min_pico=None, leyenda=None, etiqueta=None, n_regiones=None):
    """
    Atajo: capa 1 (calcular_kde_panel, con caché) + capa 2 (dibujar_panel_kde).
    Mantiene la firma que usan las funciones de figuras.
    n_regiones: None = todas las islas; N = las N de mayor densidad (REGIONES_FILA / REGIONES_MULT).
    """
    res = calcular_kde_panel(df_subset, xlims, ylims, modo_peso=modo_peso,
                             n_bootstrap=n_bootstrap, nivel_min_pico=nivel_min_pico,
                             etiqueta=etiqueta or titulo, n_regiones=n_regiones)
    return dibujar_panel_kde(ax, df_subset, res, titulo, estilos_sistema, xlims, ylims,
                             cmap=cmap, mostrar_ylabel=mostrar_ylabel,
                             lineas_extra=lineas_extra, leyenda=leyenda)


def _filas_resumen(tipo, pn, r_p):
    """Una fila por región (o una fila vacía si el panel no tiene región). Incluye masas."""
    regs = r_p["regiones"]
    if not regs:
        return [{"Tipo": tipo, "Num_Planetas": pn, "Region": np.nan, "N_regiones": 0,
                 "a_min_UA": np.nan, "sigma_a_min": np.nan,
                 "a_max_UA": np.nan, "sigma_a_max": np.nan}]
    return [{"Tipo": tipo, "Num_Planetas": pn, "Region": r["region"], "N_regiones": len(regs),
             "a_min_UA": r["a_min"], "sigma_a_min": r["s_min"],
             "a_min_menos": r["dn_min"], "a_min_mas": r["up_min"],
             "a_max_UA": r["a_max"], "sigma_a_max": r["s_max"],
             "a_max_menos": r["dn_max"], "a_max_mas": r["up_max"],
             "m_min_Mt": r.get("m_min"), "m_min_menos": r.get("dnm_min"), "m_min_mas": r.get("upm_min"),
             "m_max_Mt": r.get("m_max"), "m_max_menos": r.get("dnm_max"), "m_max_mas": r.get("upm_max"),
             "m_pico_Mt": r.get("m_pico"),
             "m_min_en_borde": r.get("m_min_en_borde"), "m_max_en_borde": r.get("m_max_en_borde")}
            for r in regs]


def _escribir_texto_regiones(res, ruta, titulo):
    """Texto con las regiones de mayor densidad: distancia y masa, con errores 1σ asimétricos."""
    def fmt(v, dn, up, d=3):
        if dn is None or up is None or pd.isna(dn) or pd.isna(up):
            return f"{v:.{d}g} (sin error estimado)"
        return f"{v:.{d}g} (+{up:.{d}g} / -{dn:.{d}g})"
    L = [titulo, "=" * len(titulo),
         "Región = isla de densidad (umbral = nivel de la barra de color inmediatamente inferior "
         "a la altura de su pico; 90.9% para el núcleo). "
         f"Errores: 1σ de las réplicas bootstrap válidas ({METODO_ERROR_REGION}).", ""]
    for _, f in res.iterrows():
        if pd.isna(f["Region"]):
            L.append(f"Tipo {f['Tipo']} | {f['Num_Planetas']} planetas: sin región definida.")
            continue
        L.append(f"Tipo {f['Tipo']} | {f['Num_Planetas']} planetas | región {int(f['Region'])} de {int(f['N_regiones'])}")
        L.append(f"   Distancia: de {fmt(f['a_min_UA'], f['a_min_menos'], f['a_min_mas'])} "
                 f"a {fmt(f['a_max_UA'], f['a_max_menos'], f['a_max_mas'])} AU")
        if pd.notna(f.get("m_min_Mt")):
            L.append(f"   Masa:      de {fmt(f['m_min_Mt'], f['m_min_menos'], f['m_min_mas'], 2)} "
                     f"a {fmt(f['m_max_Mt'], f['m_max_menos'], f['m_max_mas'], 2)} M_Tierra "
                     f"(pico en ≈ {f['m_pico_Mt']:.2f} M_Tierra)")
            if f.get("m_min_en_borde") or f.get("m_max_en_borde"):
                L.append("   ⚠ un borde de masa coincide con el límite de la malla: es una cota, no un borde real.")
        L.append("")
    texto = "\n".join(L)
    with open(ruta, "w", encoding="utf-8") as fh:
        fh.write(texto)
    print(f"Texto de regiones guardado en: {ruta}")


def plot_matriz_tipo_multiplicidad(df: pd.DataFrame, base_dir: str,
                                   cmap: str = "magma",
                                   tipos=("F", "G", "K", "M"),
                                   lineas_extra=None,
                                   modo_peso=None):
    """
    Una sola figura:
      - una fila por tipo espectral (F, G, K, M),
      - una columna por multiplicidad (cada fila usa solo las que existen),
      - barra de color única y compartida (Relative Probability Density, %),
      - isolíneas difuminadas, líneas a_min / a_max (verticales) y m_min / m_max (horizontales),
      - leyenda individual bajo cada panel.

    Devuelve un DataFrame con a_min, a_max, m_min, m_max y sus σ por panel y región.
    """
    modo_peso = modo_peso or MODO_PESO_DEFECTO
    metrica = "Area_Estable_km2"
    df = _datos_kde(df)
    if "Num_Planetas" not in df.columns:
        raise ValueError("La matriz requiere la columna Num_Planetas.")
    desconocida = df["Num_Planetas"].isna()
    if desconocida.any():
        print(f"Aviso: {int(desconocida.sum())} planetas sin multiplicidad conocida se omiten de la matriz.")
        df = df.loc[~desconocida].copy()
    if "Sistema" not in df.columns:
        df["Sistema"] = df["Planeta"].apply(lambda s: str(s).rsplit(" ", 1)[0])
    df["Tipo_Estrella"] = df["Tipo_Estrella"].astype(str).str.upper().str.strip()
    estilos = _asignar_estilos_sistema(df)

    df_f = df.loc[np.isfinite(df[metrica]) & (df[metrica] > 0)].copy()
    if len(df_f) < 3:
        print("No hay suficientes datos válidos para generar la matriz.")
        return pd.DataFrame()

    # Límites comunes a todos los paneles (incluyen Tierra y Júpiter)
    x_all = np.log10(df_f["Distancia_UA"])
    y_all = np.log10(df_f["Masa_Tierra"])
    xlims = (min(x_all.min(), 0.0) - 0.4, max(x_all.max(), np.log10(JUPITER_UA)) + 0.4)
    ylims = (min(y_all.min(), 0.0) - 0.8, max(y_all.max(), np.log10(JUPITER_MASA)) + 0.5)

    filas = []
    for tipo in tipos:
        d_t = df_f[df_f["Tipo_Estrella"] == tipo]
        if d_t.empty:
            continue
        filas.append((tipo, d_t, sorted(d_t["Num_Planetas"].unique())))
    if not filas:
        print("Ningún tipo espectral con datos.")
        return pd.DataFrame()

    nrows = len(filas)
    ncols = max(len(f[2]) for f in filas)

    # Altura de cada fila según el tamaño de su leyenda más grande
    alturas = []
    for tipo, d_t, mults in filas:
        cfg = _cfg_leyenda(tipo)
        max_filas_leyenda = 0
        for pn in mults:
            n_ent = _n_entradas_leyenda(d_t[d_t["Num_Planetas"] == pn], lineas_extra)
            max_filas_leyenda = max(max_filas_leyenda, _filas_leyenda(n_ent, cfg))
        alturas.append(6.0 + _alto_leyenda(max_filas_leyenda, cfg))

    fig = plt.figure(figsize=(ANCHO_PANEL_MATRIZ * ncols + 1.4, sum(alturas) + 1.0))
    gs = fig.add_gridspec(nrows, ncols + 1,
                          width_ratios=[1.0] * ncols + [0.05],
                          height_ratios=alturas,
                          left=0.05, right=0.95, top=0.955, bottom=0.01,
                          hspace=0.05, wspace=0.28)
    fig.suptitle("Stability Regions for Trojan Bodies by Spectral Type and Multiplicity",
                 fontsize=20, fontweight="bold", y=0.995)

    # Barra de color única y compartida
    cax = fig.add_subplot(gs[:, ncols])
    sm = ScalarMappable(norm=Normalize(0, 100), cmap=cmap)
    ticks_barra = [0.0, *sorted(NIVELES_REGIONES)]
    cbar = fig.colorbar(sm, cax=cax, ticks=ticks_barra, extend="min")
    cbar.ax.set_yticklabels([f"{v:.1f}" for v in ticks_barra], fontsize=9)
    cbar.set_label("Relative Probability Density (%)", fontsize=12, fontweight="bold")

    tareas = [(r, c, tipo, pn, d_t[d_t["Num_Planetas"] == pn].copy())
              for r, (tipo, d_t, mults) in enumerate(filas)
              for c, pn in enumerate(mults)]

    resumen = []
    filas_sigma = []
    for r, c, tipo, pn, d_sub in tqdm(tareas, desc="Generando paneles", unit="panel"):
        ax = fig.add_subplot(gs[r, c])
        r_p = _dibujar_panel_matriz(
            ax, d_sub, f"Type {tipo} Stars - {pn}-Planet Systems", estilos,
            xlims, ylims, cmap=cmap, mostrar_ylabel=(c == 0),
            lineas_extra=lineas_extra, modo_peso=modo_peso,
            leyenda=_cfg_leyenda(tipo), etiqueta=f"matriz_{tipo}_{pn}pl_{modo_peso}",
            n_regiones=REGIONES_MULT.get((tipo, int(pn))))
        resumen += _filas_resumen(tipo, pn, r_p)
        filas_sigma += _filas_sigma({"Tipo": tipo, "Num_Planetas": pn}, r_p["boot"])

    os.makedirs(base_dir, exist_ok=True)
    ruta_png = os.path.join(base_dir, f"Map_2D_Islands_Matriz_Tipo_Multiplicidad_{modo_peso}.png")
    ruta_pdf = os.path.join(base_dir, f"Map_2D_Islands_Matriz_Tipo_Multiplicidad_{modo_peso}.pdf")
    fig.savefig(ruta_png, dpi=300, bbox_inches="tight")
    fig.savefig(ruta_pdf, format="pdf", bbox_inches="tight")
    _mostrar_o_cerrar(fig)
    _guardar_tabla_sigma(filas_sigma, os.path.join(base_dir, f"intervalos_sigma_matriz_{modo_peso}.csv"))
    return pd.DataFrame(resumen)

Extracción de datos (con el área de colisión m1)

In [ ]:
# Se anclan las etiquetas a una línea y se validan las unidades de los campos.
_ETIQUETAS_AREAS = {
    "Area_Estable_km2": r"[aá]rea estable",
    "Area_Inestable": r"[aá]rea inestable",
    "Area_Colision_m1": r"[aá]rea (?:de )?colisi[oó]n (?:con )?m1",
    "Area_Colision_m2": r"[aá]rea (?:de )?colisi[oó]n (?:con )?m2",
}
_ETIQUETAS_ZONAS = {
    "Zonas Estables": r"total estables",
    "Zonas Inestables": r"total inestables",
    "Zonas Colisión m1": r"total colisi[oó]n (?:con )?m1",
    "Zonas Colisión m2": r"total colisi[oó]n (?:con )?m2",
}
_COLUMNAS_LECTURA = ["Sistema", "Planeta", "Tipo_Estrella", "Distancia_UA", "Masa_Tierra",
                    "Masa_Estrella", "Num_Planetas", *_ETIQUETAS_ZONAS, *_ETIQUETAS_AREAS]


def _numero_archivo(texto, entero=False, agrupado=True):
    """Lee decimales, exponentes y agrupaciones inequívocas; rechaza separadores ambiguos."""
    texto = texto.strip()
    signo = re.fullmatch(r"([+-]?)([\d., \u00a0\u202f]+)([eE][+-]?\d+)?", texto)
    if signo is None:
        raise ValueError(f"número inválido: {texto!r}")
    pref, cuerpo, exp = signo.groups()
    cuerpo = re.sub(r"[\u00a0\u202f]", " ", cuerpo)
    if entero:
        if exp or not re.fullmatch(r"(?:\d+|\d{1,3}(?P<sep>[., ])\d{3}(?:(?P=sep)\d{3})*)", cuerpo):
            raise ValueError(f"entero inválido: {texto!r}")
        return int(pref + re.sub(r"[., ]", "", cuerpo))
    if agrupado and "," in cuerpo and "." in cuerpo:
        decimal = "," if cuerpo.rfind(",") > cuerpo.rfind(".") else "."
        miles = "." if decimal == "," else ","
        izq, der = cuerpo.rsplit(decimal, 1)
        if not re.fullmatch(rf"\d{{1,3}}(?:{re.escape(miles)}\d{{3}})+", izq) or not der.isdigit():
            raise ValueError(f"agrupación inválida: {texto!r}")
        cuerpo = izq.replace(miles, "") + "." + der
    elif agrupado and " " in cuerpo:
        if not re.fullmatch(r"\d{1,3}(?: \d{3})+(?:[.,]\d+)?", cuerpo):
            raise ValueError(f"agrupación inválida: {texto!r}")
        cuerpo = cuerpo.replace(" ", "").replace(",", ".")
    elif agrupado and re.fullmatch(r"\d{1,3}(?P<sep>[.,])\d{3}(?:(?P=sep)\d{3})+", cuerpo):
        cuerpo = re.sub(r"[.,]", "", cuerpo)
    else:
        # Un único punto se lee como decimal (formato del proyecto). Una única coma
        # con tres cifras y sin exponente es ambigua salvo cuando su parte entera es 0.
        if agrupado and not exp and re.fullmatch(r"[1-9]\d{0,2},\d{3}", cuerpo):
            raise ValueError(f"coma ambigua en {texto!r}; use separador de miles explícito o decimal con punto")
        cuerpo = cuerpo.replace(",", ".")
    if not re.fullmatch(r"(?:\d+(?:\.\d*)?|\.\d+)", cuerpo):
        raise ValueError(f"número inválido: {texto!r}")
    valor = float(pref + cuerpo + (exp or ""))
    if not math.isfinite(valor):
        raise ValueError(f"número fuera de rango: {texto!r}")
    return valor


def _campo_resultado(texto, etiqueta, archivo, requerido=True):
    patron = rf"^[ \t]*(?:{etiqueta})[ \t]*:[ \t]*([^\r\n]*)$"
    campos = re.findall(patron, texto, flags=re.I | re.M)
    if len(campos) == 1 and campos[0].strip():
        return campos[0].strip()
    if not campos and not requerido:
        return None
    motivo = "duplicado" if len(campos) > 1 else "ausente o vacío"
    raise ValueError(f"{archivo}: campo {etiqueta!r} {motivo}")


def _numero_resultado(texto, etiqueta, archivo, *, minimo=0, entero=False, unidad=None, requerido=True):
    dato = _campo_resultado(texto, etiqueta, archivo, requerido=requerido)
    if dato is None:
        return np.nan
    try:
        if unidad:
            m = re.fullmatch(rf"(.+?)[ \t]*(?:{unidad})", dato, flags=re.I)
            if m is None:
                raise ValueError(f"unidad ausente o inválida en {dato!r}")
            dato = m[1]
        valor = _numero_archivo(dato, entero=entero)
        if valor < 0 or (minimo > 0 and valor <= 0):
            raise ValueError(f"valor no válido: {valor}")
        return valor
    except ValueError as exc:
        raise ValueError(f"{archivo}: {etiqueta}: {exc}") from exc


def _normalizar_nombre(nombre):
    return " ".join(str(nombre).replace("_", " ").split()).casefold()


def _leer_carpeta_resultados(directorio_raiz, tipo_estrella, diagnostico=None):
    """Devuelve (registros, n_sin_area_m1); guarda los motivos si se pasa diagnostico."""
    registros, avisos, descartes, n_sin_m1 = [], [], [], 0
    directorio = Path(directorio_raiz)
    if not directorio.is_dir():
        raise ValueError(f"Carpeta de resultados inexistente o inaccesible: {directorio}")
    for root, dirs, files in os.walk(directorio, onerror=lambda exc: avisos.append(str(exc))):
        dirs.sort()
        if Path(root).name in {"graficos_prediccion_c", CACHE_KDE_NOMBRE}:
            dirs.clear()
            continue
        resumenes = sorted(f for f in files if f.lower().endswith((" resumen.txt", "_resumen.txt")))
        readmes = sorted(f for f in files if f.lower().startswith("readme") and f.lower().endswith(".txt"))
        if not (resumenes or readmes):
            continue
        if len(resumenes) != 1 or len(readmes) != 1:
            descartes.append(f"{root}: se requiere un resumen y un README únicos; "
                             f"se encontró {resumenes or 'ningún resumen'} y {readmes or 'ningún README'}.")
            continue
        rr, rd = Path(root, resumenes[0]), Path(root, readmes[0])
        try:
            r, d = rr.read_text(encoding="utf-8-sig"), rd.read_text(encoding="utf-8-sig")
            sistema = _campo_resultado(d, "sistema", rd)
            planeta = _campo_resultado(d, "planeta", rd)
            distancia = _numero_resultado(d, r"semi[- ]eje mayor de la [oó]rbita \((?:ua|au)\)", rd, minimo=1)
            masa_p = _numero_resultado(d, r"masa del planeta \(masas terrestres\)", rd, minimo=1)
            masa_e = _numero_resultado(d, r"masa de la estrella \(masas solares\)", rd, minimo=1, requerido=False)
            if pd.isna(masa_e):
                avisos.append(f"{rd}: masa estelar ausente; queda NaN.")
            n_plan = _numero_resultado(d, r"n[uú]mero de planetas(?: del sistema)?", rd,
                                       minimo=1, entero=True, requerido=False)
            if _campo_resultado(r, _ETIQUETAS_AREAS["Area_Colision_m1"], rr, requerido=False) is None:
                n_sin_m1 += 1
            errores_areas, areas = [], {}
            for k, etq in _ETIQUETAS_AREAS.items():
                try:
                    areas[k] = _numero_resultado(r, etq, rr, unidad=r"km(?:²|\^?2)")
                except ValueError as exc:
                    errores_areas.append(str(exc))
            if errores_areas:
                raise ValueError(" | ".join(errores_areas))
            zonas = {}
            for k, etq in _ETIQUETAS_ZONAS.items():
                try:
                    zonas[k] = _numero_resultado(r, etq, rr, entero=True, requerido=False)
                except ValueError as exc:
                    avisos.append(str(exc))
                    zonas[k] = np.nan
            faltan_zonas = [k for k, v in zonas.items() if pd.isna(v)]
            if faltan_zonas:
                avisos.append(f"{rr}: conteos n.d. para {', '.join(faltan_zonas)}.")
            registros.append({"Sistema": sistema, "Planeta": planeta, "Tipo_Estrella": tipo_estrella,
                              "Distancia_UA": distancia, "Masa_Tierra": masa_p, "Masa_Estrella": masa_e,
                              "Num_Planetas": n_plan, **zonas, **areas, "README": str(rd), "Resumen": str(rr)})
        except (OSError, UnicodeError, ValueError) as exc:
            descartes.append(f"{rd} y {rr}: {exc}")
    if diagnostico is None:
        for motivo in avisos + descartes:
            print(f"   Aviso: {motivo}")
        print(f"    {len(descartes)} planeta(s) descartado(s) en {directorio}.")
    else:
        diagnostico["avisos"].extend(avisos)
        diagnostico["descartes"].extend(descartes)
    return registros, n_sin_m1


def _catalogo_planetas(carpetas, ruta, avisos):
    if ruta is None:
        raices = dict.fromkeys([Path.cwd(), *(p for _, d in carpetas for p in Path(d).resolve().parents)])
        rutas = (r / "b" / "Tabla_Estrellas_Seleccion_125_sistemas.csv" for r in raices)
        ruta = next((r for r in rutas if r.is_file()), None)
    if ruta is None:
        return {}
    conteos = {}
    try:
        with open(ruta, encoding="utf-8-sig", newline="") as fh:
            lector = csv.DictReader(fh)
            if not {"Host_Name", "Number_of_Planets"} <= set(lector.fieldnames or ()):
                raise ValueError("faltan columnas Host_Name y Number_of_Planets")
            for fila in lector:
                try:
                    n = int(fila["Number_of_Planets"])
                except (TypeError, ValueError):
                    continue
                if n > 0 and fila["Host_Name"]:
                    conteos.setdefault(_normalizar_nombre(fila["Host_Name"]), set()).add(n)
    except (OSError, UnicodeError, ValueError, csv.Error) as exc:
        avisos.append(f"{ruta}: catálogo de multiplicidad ilegible ({exc}).")
        return {}
    conflictos = [s for s, ns in conteos.items() if len(ns) > 1]
    if conflictos:
        avisos.append(f"{ruta}: multiplicidades contradictorias para {', '.join(conflictos)}.")
    return {s: next(iter(ns)) for s, ns in conteos.items() if len(ns) == 1}


def extraer_datos_v2(carpetas=None, *, catalogo_multiplicidad=None):
    """Lee los pares README/resumen; carpetas=None solicita rutas por diálogo."""
    if carpetas is None:
        if tk is None:
            raise RuntimeError("tkinter no disponible; usa entrada='parametros'.")
        root_tk = tk.Tk()
        try:
            root_tk.withdraw()
            root_tk.attributes("-topmost", True)
            n = simpledialog.askinteger("Cantidad de carpetas", "¿Cuántas carpetas vas a seleccionar?",
                                        minvalue=1, maxvalue=10, parent=root_tk)
            carpetas = []
            for i in range(n or 0):
                tipo = simpledialog.askstring("Tipo de Estrella", f"Tipo para la carpeta {i + 1} de {n}",
                                              parent=root_tk)
                if not tipo:
                    return pd.DataFrame(columns=_COLUMNAS_LECTURA), ""
                ruta = filedialog.askdirectory(title=f"Datos de estrellas {tipo}", parent=root_tk)
                if not ruta:
                    return pd.DataFrame(columns=_COLUMNAS_LECTURA), ""
                carpetas.append((tipo, ruta))
        finally:
            root_tk.destroy()
    carpetas = list(carpetas)
    diagnostico = {"avisos": [], "descartes": []}
    registros = []
    for tipo, ruta in carpetas:
        tipo = str(tipo).strip().upper()
        if not re.fullmatch(r"[OBAFGKM]", tipo):
            raise ValueError(f"Tipo espectral no válido para {ruta}: {tipo!r}. Use una letra O, B, A, F, G, K o M.")
        print(f"\nEscaneando {ruta} (tipo {tipo})…")
        reg, _ = _leer_carpeta_resultados(ruta, tipo, diagnostico)
        registros.extend(reg)

    df = pd.DataFrame(registros, columns=[*_COLUMNAS_LECTURA, "README", "Resumen"])
    avisos, descartes = diagnostico["avisos"], diagnostico["descartes"]
    if not df.empty:
        # Si se eligen rutas solapadas, contabiliza el mismo archivo una sola vez.
        df = df.loc[~df.apply(lambda f: (Path(f["README"]).resolve(), f["Tipo_Estrella"]), axis=1).duplicated()].copy()
        clave_s = df["Sistema"].map(_normalizar_nombre)
        clave_p = df["Planeta"].map(_normalizar_nombre)
        repetidos = pd.DataFrame({"s": clave_s, "p": clave_p}).duplicated(keep=False)
        for ruta in df.loc[repetidos, "README"]:
            descartes.append(f"{ruta}: planeta duplicado en archivos distintos o tipos espectrales distintos.")
        df = df.loc[~repetidos].copy()
        df["Num_Planetas_Leidos"] = pd.Series(index=df.index, dtype="Int64")
        df["Fuente_Num_Planetas"] = "n.d."
        catalogo = _catalogo_planetas(carpetas, catalogo_multiplicidad, avisos)
        for clave, grupo in df.groupby(df["Sistema"].map(_normalizar_nombre), sort=False):
            n_leidos = len(grupo)
            declarados = set(grupo["Num_Planetas"].dropna())
            n_cat = catalogo.get(clave)
            n, fuente = np.nan, "n.d."
            if len(declarados) == 1:
                n, fuente = next(iter(declarados)), "README"
                if n_cat is not None and n_cat != n:
                    avisos.append(f"{grupo['README'].iloc[0]}: multiplicidad README {n:g} y catálogo {n_cat}; se usa README.")
            elif not declarados and n_cat is not None:
                n, fuente = n_cat, "catálogo local"
            if pd.isna(n) or n < n_leidos:
                avisos.append(f"{grupo['README'].iloc[0]}: multiplicidad ausente o incoherente; "
                              f"se conservan {n_leidos} registros con multiplicidad n.d.")
                n, fuente = np.nan, "n.d."
            df.loc[grupo.index, "Num_Planetas"] = n
            df.loc[grupo.index, "Num_Planetas_Leidos"] = n_leidos
            df.loc[grupo.index, "Fuente_Num_Planetas"] = fuente
    df = df.sort_values(["Sistema", "Planeta"]).reset_index(drop=True)
    for motivo in avisos:
        print(f"   Aviso: {motivo}")
    for motivo in descartes:
        print(f"   Descartado: {motivo}")
    print(f"Lectura completada: {len(df)} planeta(s) válido(s), {len(descartes)} descartado(s).")
    df.attrs["diagnostico_lectura"] = diagnostico
    return df, carpetas[0][1] if carpetas else ""

Diálogo para elegir sistema, planeta b y distancia de c

In [ ]:
# ---------------------------------------------------------------------------
# DIÁLOGO: elegir sistema, planeta b y distancia de c
# ---------------------------------------------------------------------------
def _dialogo_objetivo(df):
    """Ventana para elegir el sistema, el planeta b y la distancia (UA) / masa (M_T, opcional) de c."""
    from tkinter import ttk
    sistemas = sorted(df["Sistema"].unique())
    root = tk.Tk()
    root.title("Predicción del planeta c")
    root.attributes("-topmost", True)
    resultado = {}
    v_sis, v_b, v_ac, v_mc, v_info = (tk.StringVar() for _ in range(5))

    ttk.Label(root, text="Sistema:").grid(row=0, column=0, sticky="w", padx=8, pady=4)
    cb_sis = ttk.Combobox(root, textvariable=v_sis, values=sistemas, state="readonly", width=42)
    cb_sis.grid(row=0, column=1, padx=8, pady=4)
    ttk.Label(root, text="Planeta b:").grid(row=1, column=0, sticky="w", padx=8, pady=4)
    cb_b = ttk.Combobox(root, textvariable=v_b, state="readonly", width=42)
    cb_b.grid(row=1, column=1, padx=8, pady=4)
    ttk.Label(root, textvariable=v_info, foreground="gray").grid(row=2, column=0, columnspan=2, padx=8)
    ttk.Label(root, text="Distancia de c (UA):").grid(row=3, column=0, sticky="w", padx=8, pady=4)
    ttk.Entry(root, textvariable=v_ac, width=20).grid(row=3, column=1, sticky="w", padx=8, pady=4)
    ttk.Label(root, text="Masa de c (M_Tierra, opcional):").grid(row=4, column=0, sticky="w", padx=8, pady=4)
    ttk.Entry(root, textvariable=v_mc, width=20).grid(row=4, column=1, sticky="w", padx=8, pady=4)

    def _actualiza(*_):
        d = df[df["Sistema"] == v_sis.get()]
        pls = list(d["Planeta"])
        cb_b["values"] = pls
        candidatos_b = [p for p in pls if _normalizar_nombre(p).split()[-1:] == ["b"]]
        v_b.set(candidatos_b[0] if len(candidatos_b) == 1 else "")
        if len(d):
            v_info.set(f"Tipo {d['Tipo_Estrella'].iloc[0]} | {len(d)} planetas con datos en este sistema")

    def _aceptar():
        try:
            ac = _numero_archivo(v_ac.get(), agrupado=False)
            mc = _numero_archivo(v_mc.get(), agrupado=False) if v_mc.get().strip() else None
            if ac <= 0 or (mc is not None and mc <= 0) or not v_sis.get() or not v_b.get():
                raise ValueError
        except ValueError:
            messagebox.showerror("Datos inválidos",
                                 "Elige sistema y planeta b e ingresa una distancia de c > 0 "
                                 "(y una masa > 0 si la pones).", parent=root)
            return
        resultado.update({"sistema": v_sis.get(), "planeta_b": v_b.get(), "a_c": ac, "m_c": mc})
        root.destroy()

    cb_sis.bind("<<ComboboxSelected>>", _actualiza)
    ttk.Button(root, text="Aceptar", command=_aceptar).grid(row=5, column=0, columnspan=2, pady=10)
    root.protocol("WM_DELETE_WINDOW", root.destroy)
    root.mainloop()
    return resultado or None

Núcleo: KDE de un punto, evaluación de c y figuras

In [ ]:
# ---------------------------------------------------------------------------
# NÚCLEO: KDE de un solo punto, evaluación de c y figuras
# ---------------------------------------------------------------------------
BORDES_BARRA = np.array([0.0, *sorted(NIVELES_REGIONES), 100.0])
FUENTES_ERROR = ("observacional", "ancho", "bootstrap_sistemas", "bootstrap_filas")


def _covarianza_valida(cov):
    cov = np.asarray(cov, dtype=float)
    if cov.shape != (2, 2) or not np.isfinite(cov).all() or not np.allclose(cov, cov.T, rtol=1e-10, atol=0):
        raise ValueError("cov debe ser simétrica y finita de 2 × 2.")
    cov = cov / 2 + cov.T / 2
    try:
        np.linalg.cholesky(cov)
    except np.linalg.LinAlgError as exc:
        raise ValueError("cov debe ser definida positiva.") from exc
    return cov


def _radios_gauss(cov, nivel):
    # En 90.9 %, el radio proyectado es 0.437 veces la desviación del kernel.
    q = math.sqrt(-2.0 * math.log(nivel / 100.0))
    return q * np.sqrt(np.diagonal(cov, axis1=-2, axis2=-1))


def _sigma_observacional(log_value, sigma, nombre):
    if sigma is None:
        return None
    sigma = float(sigma)
    if not math.isfinite(sigma) or sigma < 0:
        raise ValueError(f"{nombre} debe ser finito y no negativo.")
    # Lognormal con mediana central y coeficiente de variación sigma/value.
    if sigma == 0:
        return 0.0
    log_ratio = math.log(sigma) - math.log(10.0) * log_value
    return math.sqrt(np.logaddexp(0.0, 2.0 * log_ratio)) / math.log(10.0)


def _bordes_gauss(centros, radios):
    a, m = np.moveaxis(centros, -1, 0)
    ra, rm = np.moveaxis(radios, -1, 0)
    return dict(zip(LIMITES_REGION, (a - ra, a + ra, m - rm, m + rm)))


def _remuestrear_cov(d, modo_peso, n, rng):
    """Reajusta el ancho KDE remuestreando sistemas completos."""
    x = np.log10(d["Distancia_UA"].to_numpy())
    y = np.log10(d["Masa_Tierra"].to_numpy())
    w = _calcular_pesos(d, modo_peso)
    grupos = [np.asarray(i) for i in d.groupby("Sistema", sort=False).indices.values()]
    if len(grupos) < 2:
        return np.empty((0, 2, 2))
    covs = []
    for sel in rng.integers(0, len(grupos), size=(n, len(grupos))):
        idx = np.concatenate([grupos[i] for i in sel])
        kde = _ajustar_kde(x[idx], y[idx], w[idx], bw=KDE_BW)
        if kde is not None:
            covs.append(kde.covariance)
    return np.asarray(covs).reshape(-1, 2, 2)


def calcular_kde_solo_b(x_b, y_b, cov, xlims, ylims, nivel_min=None, *,
                       n_bootstrap=None, sigma_a_b=None, sigma_m_b=None,
                       sigma_log_ancho=None, covs_boot=None, fuente_cov="covarianza suministrada",
                       cache_dir=None):
    """
    Un punto gaussiano con pico analítico de 100 %. El peso de b se cancela.
    Los bordes dependen del ancho del kernel. Se obtienen sin cuantización de malla.
    Propaga covarianzas remuestreadas o sensibilidad al ancho, más la contribución
    observacional especificada. None indica incertidumbre observacional desconocida.
    """
    _validar_malla(xlims, ylims)
    cov = _covarianza_valida(cov)
    centro = np.asarray([x_b, y_b], dtype=float)
    if not np.isfinite(centro).all():
        raise ValueError("Las coordenadas logarítmicas de b deben ser finitas.")
    nb = _n_replicas(n_bootstrap, PRED["n_bootstrap"])
    nmp = NIVEL_MIN_PICO if nivel_min is None else float(nivel_min)
    nivel = _nivel_isla(100.0) if NIVELES_POR_ALTURA else NIVEL_REGION_ESTABLE
    if not 0 <= nmp <= 100 or nivel is None or not 0 < nivel <= 100:
        raise ValueError("Nivel de pico o nivel de contorno inválido.")
    s_obs = (_sigma_observacional(x_b, sigma_a_b, "sigma_a_b"),
             _sigma_observacional(y_b, sigma_m_b, "sigma_m_b"))
    s_ancho = float(PRED["sigma_log_ancho"] if sigma_log_ancho is None else sigma_log_ancho)
    if not math.isfinite(s_ancho) or s_ancho < 0:
        raise ValueError("sigma_log_ancho debe ser finito y no negativo.")
    covs = np.empty((0, 2, 2)) if covs_boot is None else np.asarray(covs_boot, dtype=float)
    if covs.ndim != 3 or covs.shape[1:] != (2, 2):
        raise ValueError("covs_boot debe tener forma (n, 2, 2).")
    covs = covs[:nb]
    clave = _hash_cache(centro, cov, tuple(xlims), tuple(ylims), nmp, nivel, nb, s_obs, s_ancho, covs,
                        fuente_cov, sigma_a_b, sigma_m_b, KDE_GRID_RES, SEMILLA_BOOTSTRAP, METODO_ERROR_REGION)

    def _calc():
        radios = _radios_gauss(cov, nivel)
        X, Y = np.mgrid[xlims[0]:xlims[1]:KDE_GRID_RES, ylims[0]:ylims[1]:KDE_GRID_RES]
        # Añade nodos cerca de b si la malla global no resuelve el núcleo.
        ejes = []
        for eje, media, sd in zip((X[:, 0], Y[0, :]), centro, np.sqrt(np.diag(cov))):
            if eje[1] - eje[0] > sd / 4:
                locales = media + sd * np.linspace(-3, 3, 61)
                eje = np.unique(np.r_[eje, locales[(eje[0] < locales) & (locales < eje[-1])]])
            ejes.append(eje)
        X, Y = np.meshgrid(*ejes, indexing="ij")
        Z_pct = 100.0 * _kde_cov_fija([x_b], [y_b], [1.0], cov, X, Y)
        res = {"malla": (X, Y, Z_pct), "regiones": [], "boot": {}, "avisos": [],
               "fuente_cov": fuente_cov, "cov": cov,
               "modelo_solo_b": (float(x_b), float(y_b), np.linalg.inv(cov)),
               "sigma_a_b": sigma_a_b, "sigma_m_b": sigma_m_b, "sigma_log_ancho": s_ancho}
        nodos = Z_pct >= nivel
        bordes = _bordes_gauss(centro, radios)
        nominales = tuple(float(bordes[k]) for k in LIMITES_REGION)
        mesh = tuple(float(v) for v in (X[nodos].min(), X[nodos].max(), Y[nodos].min(), Y[nodos].max())) \
            if nodos.any() else (np.nan,) * 4
        semilla_obs, semilla_ancho = np.random.SeedSequence(SEMILLA_BOOTSTRAP).spawn(2)
        fuentes = {}
        if any(s is not None for s in s_obs):
            rng_obs = np.random.default_rng(semilla_obs)
            desplazamientos = rng_obs.normal(size=(nb, 2)) * np.asarray([s or 0.0 for s in s_obs])
            obs = _bordes_gauss(centro + desplazamientos, radios)
            estimados = [eje for eje, s in zip(("a", "m"), s_obs) if s is not None]
            fuentes["observacional"] = {k: v for k, v in obs.items() if k[0] in estimados}
        else:
            desplazamientos = np.zeros((nb, 2))
        if any(s is None for s in s_obs):
            faltan = [n for n, s in zip(("sigma_a_b", "sigma_m_b"), s_obs) if s is None]
            res["avisos"].append("Sin error observacional estimado para " + ", ".join(faltan) + ".")
        if len(covs) >= 5:
            if not np.isfinite(covs).all() or (np.diagonal(covs, axis1=-2, axis2=-1) <= 0).any():
                raise ValueError("Covarianzas remuestreadas inválidas.")
            radios_rep = _radios_gauss(covs, nivel)
            fuente = "bootstrap_sistemas"
        else:
            rng_ancho = np.random.default_rng(semilla_ancho)
            radios_rep = radios * np.exp(rng_ancho.normal(0.0, s_ancho, size=(nb, 1)))
            fuente = "ancho"
            if nb and fuente_cov.startswith("referencia"):
                res["avisos"].append("Covarianza heredada con menos de cinco réplicas válidas. "
                                     "El ancho se evalúa mediante sensibilidad lognormal.")
        nr = len(radios_rep)
        if nb:
            fuentes[fuente] = _bordes_gauss(centro, radios_rep)
        fuentes["total"] = _bordes_gauss(centro + desplazamientos[:nr], radios_rep)
        res["fuente_incertidumbre"] = fuente + (" + observacional" if "observacional" in fuentes else "")
        if fuente_cov == "valor por defecto":
            res["avisos"].append("Covarianza de solo_b por defecto. La elipse describe el ancho supuesto del kernel.")
        _registrar_region(res, (*nominales, y_b), {"nodos": mesh, "nivel": nivel, "pico": (x_b, y_b)}, fuentes)
        return res

    return _cache_obtener("solo_b", clave, _calc, cache_dir if SEMILLA_BOOTSTRAP is not None else None)


def _densidad_en_c(malla, lx_c, ly_c=None):
    """Densidad bilineal en c o máximo sobre masa. Fuera de malla devuelve (NaN, None)."""
    if malla is None:
        return np.nan, None
    X, Y, Z = malla
    xs, ys = X[:, 0], Y[0, :]
    if not xs[0] <= lx_c <= xs[-1] or (ly_c is not None and not ys[0] <= ly_c <= ys[-1]):
        return np.nan, None
    interp = RegularGridInterpolator((xs, ys), Z, bounds_error=False)
    if ly_c is not None:
        return float(interp([[lx_c, ly_c]])[0]), float(ly_c)
    v = interp(np.column_stack([np.full_like(ys, lx_c), ys]))
    j = int(v.argmax())
    return float(v[j]), float(ys[j])


def _densidad_analitica_solo_b(res, lx, ly):
    x, y, inv = res["modelo_solo_b"]
    dx = lx - x
    # Con masa desconocida, maximiza la gaussiana sobre log(m).
    dy = ly - y if ly is not None else -inv[1, 0] / inv[1, 1] * dx
    d = np.array([dx, dy])
    return 100.0 * math.exp(-0.5 * (d @ inv @ d))


def _en_isla_interpolada(malla, r, lx, ly, dens):
    """Pertenencia local a la componente, aunque otra isla supere su nivel."""
    if not np.isfinite(dens):
        return np.nan
    if dens < r["nivel"]:
        return False
    comp = r.get("comp")
    if comp is None:
        return True  # solo_b tiene una sola isla.
    X, Y, Z = malla
    xs, ys = X[:, 0], Y[0, :]
    i = int(np.clip(np.searchsorted(xs, lx, side="right") - 1, 0, len(xs) - 2))
    u = (lx - xs[i]) / (xs[i + 1] - xs[i])
    if ly is None:
        perfil = (1 - u) * Z[i] + u * Z[i + 1]
        return bool(np.any((perfil >= r["nivel"]) & (comp[i] | comp[i + 1])))
    j = int(np.clip(np.searchsorted(ys, ly, side="right") - 1, 0, len(ys) - 2))
    v = (ly - ys[j]) / (ys[j + 1] - ys[j])
    sub = Z[i:i + 2, j:j + 2]
    # Decide por trayectorias por cada eje que conservan el nivel.
    ax_x = (1 - v) * sub[:, 0] + v * sub[:, 1]
    ax_y = (1 - u) * sub[0, :] + u * sub[1, :]
    alcanzables = (ax_x[:, None] >= r["nivel"]) | (ax_y[None, :] >= r["nivel"])
    return bool(np.any(alcanzables & comp[i:i + 2, j:j + 2]))


def _intervalo_barra(dens):
    if not np.isfinite(dens):
        return "n.d."
    bordes = [0.0, *sorted(NIVELES_REGIONES), 100.0]
    i = int(np.clip(np.searchsorted(bordes, dens, side="right") - 1, 0, len(bordes) - 2))
    return f"{bordes[i]:.1f}-{bordes[i + 1]:.1f}"


def _cobertura_log_a(regs, rango):
    """Fracción geométrica del rango de referencia, con línea base uniforme en log(a)."""
    if rango is None:
        return np.nan
    lo0, hi0 = rango
    if not np.isfinite([lo0, hi0]).all() or hi0 <= lo0:
        return np.nan
    iv = sorted((max(r["lmin"], lo0), min(r["lmax"], hi0)) for r in regs
                if r["lmax"] > lo0 and r["lmin"] < hi0)
    tot, fin = 0.0, lo0
    for a, b in iv:
        if b > max(a, fin):
            tot += b - max(a, fin)
            fin = b
    return tot / (hi0 - lo0)


def _evaluar_c(res, lx_c, ly_c, rango_ref):
    """Evalúa la proyección radial y, con masa, la pertenencia al contorno de la isla."""
    dens, ly_dens = _densidad_en_c(res["malla"], lx_c, ly_c)
    analitico = "modelo_solo_b" in res
    dens_gauss = _densidad_analitica_solo_b(res, lx_c, ly_c) if analitico else np.nan
    filas = []
    for r in res["regiones"]:
        k = r["region"]
        lo, hi = r["lmin"], r["lmax"]
        margen = min(lx_c - lo, hi - lx_c)
        proyeccion = bool(margen >= -1e-12)
        por_densidad = _en_isla_interpolada(res["malla"], r, lx_c, ly_c, dens)
        masa_ok = np.nan if ly_c is None else bool(r["log_m_min"] <= ly_c <= r["log_m_max"])
        if ly_c is None:
            nominal, criterio = proyeccion, "proyección exacta en log(a)"
        elif analitico:
            nominal = bool(proyeccion and dens_gauss >= r["nivel"] - 1e-10)
            criterio = "contorno gaussiano 2D y proyección en log(a)"
        else:
            nominal = bool(proyeccion and pd.notna(por_densidad) and por_densidad)
            criterio = "isla interpolada 2D y proyección en log(a)"
        fila = {"Region": k, "umbral_isla_%": r["nivel"],
                "a_min_UA": r["a_min"], "a_max_UA": r["a_max"], "m_min_Mt": r["m_min"], "m_max_Mt": r["m_max"],
                "a_min_err": r["s_min"], "a_max_err": r["s_max"], "masa_c_en_region": masa_ok,
                "c_en_region_nominal": nominal, "criterio_decision": criterio,
                "criterio_bordes_c": proyeccion, "criterio_densidad_c": por_densidad,
                "c_supera_umbral_interpolado": bool(dens >= r["nivel"]) if np.isfinite(dens) else np.nan,
                "criterios_coinciden": bool(proyeccion == por_densidad) if pd.notna(por_densidad) else np.nan,
                "interpolacion_coincide_con_nominal": bool(nominal == por_densidad) if pd.notna(por_densidad) else np.nan,
                "delta_dex_al_borde": float(max(0.0, -margen)), "margen_proyeccion_dex": float(margen)}
        requeridos = LIMITES_REGION[:2] if ly_c is None else LIMITES_REGION
        n_replicas = []
        intervalos = {}
        for nombre, nodo in zip(LIMITES_REGION, r["nodos"]):
            unidad = "UA" if nombre.startswith("a_") else "Mt"
            central, replicas = res["boot"][f"{nombre}_{k}"]
            fila[f"{nombre}_exacto_{unidad}"] = r[nombre]
            fila[f"{nombre}_malla_{unidad}"] = _desde_log10(nodo)
            fila[f"{nombre}_acotado_malla"] = bool(r[f"{nombre}_en_borde"] and not analitico)
            n_validas = int(np.isfinite(replicas).sum())
            fila[f"{nombre}_n_replicas"] = n_validas
            if nombre in requeridos:
                n_replicas.append(n_validas)
            for fuente in ("total", *FUENTES_ERROR):
                vals = r["fuentes"].get(fuente, {}).get(nombre, [])
                dn, up = _error_asimetrico(r[nombre], vals)
                fila[f"{nombre}_{fuente}_err_menos_{unidad}"] = dn
                fila[f"{nombre}_{fuente}_err_mas_{unidad}"] = up
            for sigma in (1, 3):
                iv = _intervalo_sigma(replicas, sigma, central)
                intervalos[nombre, sigma] = iv
                for lado, i in (("inf", 0), ("sup", 1)):
                    fila[f"{nombre}_{sigma}sigma_{lado}_{unidad}"] = _desde_log10(iv[i]) if iv is not None else np.nan
        fila["N_replicas_borde_min"] = min(n_replicas)
        fila["error_estimado"] = min(n_replicas) >= 5 and all(
            np.isfinite(fila[f"{nombre}_total_err_{lado}_{'UA' if nombre.startswith('a_') else 'Mt'}"])
            for nombre in requeridos for lado in ("menos", "mas"))
        fila["Estado_error"] = "estimado" if fila["error_estimado"] else "sin error estimado"
        for sigma in (1, 3):
            # Envolvente de cuantiles marginales; con masa se verifica también ese eje.
            acepta = True
            for coord, valor in (("a", lx_c), ("m", ly_c)):
                if valor is None:
                    continue
                inf, sup = (intervalos[f"{coord}_{lado}", sigma] for lado in ("min", "max"))
                if inf is None or sup is None:
                    acepta = np.nan
                    break
                acepta &= bool(inf[0] <= valor <= sup[1])
            fila[f"c_en_region_{sigma}sigma"] = acepta
        filas.append(fila)
    cobertura = _cobertura_log_a(res["regiones"], rango_ref) if res["malla"] is not None else np.nan
    info = {"Densidad_rel_en_c_%": dens, "Densidad_analitica_en_c_%": dens_gauss,
            "Intervalo_barra_c_%": _intervalo_barra(dens) if np.isfinite(dens) else "n.d.",
            "Masa_log_donde_max": ly_dens, "Cobertura_log_a_%": 100.0 * cobertura,
            "N_regiones": len(filas), "c_en_malla": bool(np.isfinite(dens))}
    return filas, info


def _formato_num(valor, cifras=5):
    return f"{valor:.{cifras}g}" if pd.notna(valor) else "n.d."


def _si_no(valor):
    return "n.d." if pd.isna(valor) else ("sí" if valor else "no")


def _diagnostico_region(f, compacto=False):
    """Valores y criterios compartidos por TXT y figuras."""
    if pd.isna(f.get("Region")):
        return ["Sin región estimable"]
    k = int(f["Region"])
    lineas = [
        f"Región {k} · {f['umbral_isla_%']:.4g} % · "
        f"{f['criterio_decision']} · nominal {_si_no(f['c_en_region_nominal'])}",
        f"Proyección {_si_no(f['criterio_bordes_c'])} · Densidad/isla {_si_no(f['criterio_densidad_c'])} · "
        f"Coinciden {_si_no(f['criterios_coinciden'])} · 1σ {_si_no(f['c_en_region_1sigma'])} · "
        f"3σ {_si_no(f['c_en_region_3sigma'])} · {f['Estado_error']}",
    ]
    if not compacto:
        lineas.append(f"Distancia radial exterior al borde = {f['delta_dex_al_borde']:.5g} dex · "
                      f"réplicas por borde ≥ {f['N_replicas_borde_min']}.")
        if f["interpolacion_coincide_con_nominal"] is False:
            lineas.append("La pertenencia interpolada difiere de la decisión nominal.")
        if any(f[f"{k}_acotado_malla"] for k in LIMITES_REGION):
            lineas.append("Al menos un borde de la isla es una cota de la malla.")
    for eje, unidad in (("a", "UA"), ("m", "Mt")):
        limites = [f"{eje}_min", f"{eje}_max"]
        exactos = "–".join(_formato_num(f[f"{k}_exacto_{unidad}"]) for k in limites)
        nodos = "–".join(_formato_num(f[f"{k}_malla_{unidad}"]) for k in limites)
        detalle = []
        for fuente in ("total", *FUENTES_ERROR):
            pares = [(f[f"{k}_{fuente}_err_menos_{unidad}"], f[f"{k}_{fuente}_err_mas_{unidad}"]) for k in limites]
            if compacto and fuente in {"bootstrap_sistemas", "bootstrap_filas"} and not np.isfinite(pares).any():
                continue
            if not np.isfinite(pares).any():
                detalle.append(f"{fuente} n.d.")
                continue
            errores = ", ".join(f"−{_formato_num(dn, 2)}/+{_formato_num(up, 2)}" for dn, up in pares)
            detalle.append(f"{fuente} {errores}")
        lineas.append(f"{eje} ({unidad}) · contorno {exactos} · nodos {nodos}")
        lineas.append("   1σ " + " | ".join(detalle))
    return lineas


def _anotar_diagnostico(fig, filas, mapa=False):
    if filas is None:
        return
    bloques = ["\n".join(_diagnostico_region(f, compacto=True)) for f in filas]
    if not bloques:
        bloques = ["Sin región estimable"]
    # La leyenda original queda sobre el resumen de diagnóstico.
    fig.text(0.12, -0.10 if mapa else -0.12, "\n\n".join(bloques), va="top", fontsize=8)


def _figura_prediccion(res, d_ref, fila_b, sistema, tipo, a_c, m_c, titulo_modo, xlims, ylims, ruta, cmap="magma", filas=None):
    """Mapa de estabilidad + b + línea de c. Reutiliza dibujar_panel_kde (capa de diseño)."""
    etq_ref = f"Reference type {tipo} (n={len(d_ref)})"
    etq_b = f"{fila_b['Planeta']} (b)"
    d_plot = d_ref.copy()
    d_plot["Sistema"] = etq_ref
    d_b = pd.DataFrame([fila_b])
    d_b["Sistema"] = etq_b
    d_all = pd.concat([d_plot, d_b], ignore_index=True) if not d_ref.empty else d_b
    estilos = {etq_ref: {"marker": "o", "color": "lightsteelblue"}, etq_b: {"marker": "*", "color": "gold"}}

    fig, ax = plt.subplots(figsize=(10.5, 11.5))
    dibujar_panel_kde(ax, d_all, res, f"{sistema} (type {tipo}) - {titulo_modo}", estilos, xlims, ylims,
                      cmap=cmap, lineas_extra=[("Planet c", a_c, "cyan", ":")],
                      leyenda={"fs": 10, "ms": 8, "ncol": 2, "y": -0.15, "fs_cajas": 9.5})
    lb, mb = math.log10(fila_b["Distancia_UA"]), math.log10(fila_b["Masa_Tierra"])
    ax.scatter(lb, mb, marker="*", s=330, color="gold", edgecolor="black", linewidth=0.9, zorder=11)
    ax.annotate("b", (lb, mb), xytext=(8, 8), textcoords="offset points", color="white",
                fontsize=13, fontweight="bold", zorder=12)
    lc = math.log10(a_c)
    if m_c is not None:
        ax.scatter(lc, math.log10(m_c), marker="D", s=120, color="cyan", edgecolor="black",
                   linewidth=0.9, zorder=11)
        ax.annotate("c", (lc, math.log10(m_c)), xytext=(8, 8), textcoords="offset points",
                    color="cyan", fontsize=13, fontweight="bold", zorder=12)
    else:
        ax.annotate("c", xy=(lc, ylims[1]), xytext=(4, -6), textcoords="offset points", ha="left",
                    va="top", color="cyan", fontsize=13, fontweight="bold", zorder=12)
    _anotar_diagnostico(fig, filas, mapa=True)
    sm = ScalarMappable(norm=Normalize(0, 100), cmap=cmap)
    ticks_barra = [0.0, *sorted(NIVELES_REGIONES)]
    cbar = fig.colorbar(sm, ax=ax, ticks=ticks_barra, extend="min", pad=0.02, fraction=0.046)
    cbar.ax.set_yticklabels([f"{v:.1f}" for v in ticks_barra], fontsize=9)
    cbar.set_label("Relative KDE density (%)", fontsize=12, fontweight="bold")
    fig.savefig(ruta + ".png", dpi=300, bbox_inches="tight")
    fig.savefig(ruta + ".pdf", format="pdf", bbox_inches="tight")
    _mostrar_o_cerrar(fig)


def _figura_perfil(res, a_b, a_c, titulo, ruta, filas=None):
    """Densidad máxima (sobre todas las masas) en función de la distancia, con las regiones, b y c."""
    if res["malla"] is None:
        return
    X, Y, Z = res["malla"]
    fig, ax = plt.subplots(figsize=(9.5, 4.2))
    ax.plot(X[:, 0], Z.max(axis=1), color="black", lw=1.8)
    ax.fill_between(X[:, 0], 0, Z.max(axis=1), color="0.85")
    for r in res["regiones"]:
        col = COLORES_REGIONES[(r["region"] - 1) % len(COLORES_REGIONES)]
        ax.axvspan(r["lmin"], r["lmax"], color=col, alpha=0.18)
        for nodo in r["nodos"][:2]:
            if np.isfinite(nodo):
                ax.axvline(nodo, color=col, ls="--", lw=0.9, alpha=0.8)
    ax.axvline(math.log10(a_b), color="goldenrod", lw=2, label=f"b ({a_b:.5g} AU)")
    ax.axvline(math.log10(a_c), color="deepskyblue", lw=2, ls=":", label=f"c ({a_c:.5g} AU)")
    ax.set_xlabel("log[Distance] (AU)", fontsize=12)
    ax.set_ylabel("max density over mass (%)", fontsize=12)
    ax.set_title(titulo, fontsize=12, fontweight="bold")
    lx_c = math.log10(a_c)
    ax.set_xlim(min(X[0, 0], lx_c - 0.04), max(X[-1, 0], lx_c + 0.04))
    ax.set_ylim(0, 102)
    ax.grid(True, ls="--", alpha=0.3)
    ax.legend(loc="upper right", fontsize=10)
    _anotar_diagnostico(fig, filas)
    fig.savefig(ruta + ".png", dpi=200, bbox_inches="tight")
    fig.savefig(ruta + ".pdf", bbox_inches="tight")
    _mostrar_o_cerrar(fig)

Predicción de c

In [ ]:
# Predicción de c y exportación
def _fila_planeta(df, sistema, planeta):
    """Nombre completo o sufijo de una sola letra orbital, con coincidencia única."""
    d = df.loc[df["Sistema"].map(_normalizar_nombre) == _normalizar_nombre(sistema)]
    nombre = _normalizar_nombre(planeta)
    nombres = d["Planeta"].map(_normalizar_nombre)
    f = d.loc[nombres == nombre]
    if f.empty and re.fullmatch(r"[a-z]", nombre):
        f = d.loc[nombres.map(lambda n: n.split()[-1:] == [nombre])]
    if len(f) > 1:
        raise ValueError(f"Planeta ambiguo {planeta!r} en {sistema!r}: {', '.join(f['Planeta'])}.")
    return None if f.empty else f.iloc[0]


def _valor_positivo(v, nombre):
    try:
        v = float(v)
    except (TypeError, ValueError) as exc:
        raise ValueError(f"{nombre} debe ser finito y mayor que cero.") from exc
    if not math.isfinite(v) or v <= 0:
        raise ValueError(f"{nombre} debe ser finito y mayor que cero.")
    return v


def _calibrar_solo_b(d_ref, d_ext, modo_peso, anchos, nb, avisos):
    for fuente, d in (("referencia del mismo tipo", d_ref), ("referencia de todos los tipos", d_ext)):
        if d["Sistema"].nunique() < 2:
            continue
        x, y = (np.log10(d[c].to_numpy()) for c in ("Distancia_UA", "Masa_Tierra"))
        kde = _ajustar_kde(x, y, _calcular_pesos(d, modo_peso), bw=KDE_BW)
        if kde is not None:
            if fuente.endswith("todos los tipos"):
                avisos.append("Kernel de solo_b heredado de todos los tipos espectrales disponibles.")
            covs = _remuestrear_cov(d, modo_peso, nb, np.random.default_rng(SEMILLA_BOOTSTRAP))
            return kde.covariance, covs, fuente
    return np.diag(np.square(anchos)), None, (
        "valor por defecto" if tuple(anchos) == (0.35, 0.80) else "anchos de respaldo configurados")


def predecir_c(df, sistema, planeta_b, a_c, m_c=None, modo_region=None, modo_peso=None,
               n_bootstrap=None, nivel_min_pico=None, cov_solo_b_dex=None, base_dir=None, cmap="magma",
               *, sigma_a_b=None, sigma_m_b=None, sigma_log_ancho=None):
    """Predice regiones y devuelve un DataFrame con una fila por modo y región."""
    modos = {"referencia": ("referencia",), "solo_b": ("solo_b",), "ambos": ("referencia", "solo_b")}
    modo_region = PRED["modo_region"] if modo_region is None else modo_region
    if modo_region not in modos:
        raise ValueError(f"modo_region desconocido: {modo_region!r}.")
    modo_peso = PRED["modo_peso"] if modo_peso is None else modo_peso
    nb = _n_replicas(n_bootstrap, PRED["n_bootstrap"])
    a_c = _valor_positivo(a_c, "a_c")
    m_c = _valor_positivo(m_c, "m_c") if m_c is not None else None
    lx_c, ly_c = math.log10(a_c), math.log10(m_c) if m_c is not None else None
    faltan = {"Tipo_Estrella", "Sistema", "Planeta", "Distancia_UA", "Masa_Tierra"} - set(df.columns)
    if faltan:
        raise ValueError(f"Faltan columnas: {', '.join(sorted(faltan))}.")
    fila_b = _fila_planeta(df, sistema, planeta_b)
    if fila_b is None:
        disponibles = df.loc[df["Sistema"].map(_normalizar_nombre) == _normalizar_nombre(sistema), "Planeta"]
        print(f"Planeta {planeta_b!r} ausente en {sistema!r}. Disponibles: {', '.join(disponibles) or 'ninguno'}.")
        return None
    sistema, planeta_b = fila_b["Sistema"], fila_b["Planeta"]
    tipo = str(fila_b["Tipo_Estrella"]).strip().upper()
    if not re.fullmatch(r"[OBAFGKM]", tipo):
        raise ValueError(f"Tipo espectral inválido para {planeta_b}: {tipo!r}.")
    a_b = _valor_positivo(fila_b["Distancia_UA"], "a_b")
    m_b = _valor_positivo(fila_b["Masa_Tierra"], "m_b")
    x_b, y_b = math.log10(a_b), math.log10(m_b)
    m_estrella = pd.to_numeric(fila_b.get("Masa_Estrella", np.nan), errors="coerce")
    m_estrella = float(m_estrella) if pd.notna(m_estrella) and np.isfinite(m_estrella) and m_estrella > 0 else np.nan
    sigma_a_b = PRED["sigma_a_b"] if sigma_a_b is None else sigma_a_b
    sigma_m_b = PRED["sigma_m_b"] if sigma_m_b is None else sigma_m_b

    # c y el resto del sistema objetivo quedan excluidos de toda calibración.
    fuera = df["Sistema"].notna() & (df["Sistema"].map(_normalizar_nombre) != _normalizar_nombre(sistema))
    d_ext = _datos_kde(df.loc[fuera])
    d_ext["Sistema"] = d_ext["Sistema"].map(_normalizar_nombre)
    d_ext["Tipo_Estrella"] = d_ext["Tipo_Estrella"].astype(str).str.strip().str.upper()
    w = _calcular_pesos(d_ext, modo_peso)
    d_ext = d_ext.loc[np.isfinite(w) & (w > 0) & d_ext["Tipo_Estrella"].str.fullmatch(r"[OBAFGKM]")].copy()
    d_ref = d_ext[d_ext["Tipo_Estrella"] == tipo]
    n_sis_ref = d_ref["Sistema"].nunique()
    xref = np.log10(d_ref["Distancia_UA"].to_numpy())
    yref = np.log10(d_ref["Masa_Tierra"].to_numpy())
    rango_ref = (float(xref.min()), float(xref.max())) if xref.size > 1 and np.ptp(xref) > 0 else None
    # El punto c puede ampliar la figura, pero no modifica la malla del modelo.
    xs = [*xref, x_b, 0.0, math.log10(JUPITER_UA)]
    ys = [*yref, y_b, 0.0, math.log10(JUPITER_MASA)]
    xlims = (min(xs) - 0.4, max(xs) + 0.4)
    ylims = (min(ys) - 0.8, max(ys) + 0.5)
    w_b = _calcular_pesos(pd.DataFrame([fila_b]), modo_peso)[0]
    avisos = []
    if len(d_ext) < int(fuera.sum()):
        avisos.append(f"{int(fuera.sum()) - len(d_ext)} planeta(s) externos con datos inválidos o peso no positivo.")
    if d_ref.empty:
        avisos.append("Referencia del mismo tipo vacía. La cobertura en log(a) queda n.d.")
    elif rango_ref is None:
        avisos.append("Referencia sin amplitud en log(a). La cobertura queda n.d.")
    if pd.isna(m_estrella):
        avisos.append("Masa de la estrella objetivo n.d.")
    if not np.isfinite(w_b):
        avisos.append("Peso de b n.d. por áreas ausentes o inválidas.")
    print(f"\nPredicción de c | {sistema} | tipo {tipo} | b = {planeta_b} "
          f"(a = {a_b:.5g} UA, m = {m_b:.5g} Mt, peso = {_formato_num(w_b)})")
    print(f"Referencia: {len(d_ref)} planeta(s) de {n_sis_ref} sistema(s).")

    base_dir = Path(base_dir if base_dir is not None else "graficos_prediccion_c")
    base_dir.mkdir(parents=True, exist_ok=True)
    cache_dir = _ruta_cache(base_dir)
    suf = re.sub(r"[^\w\-]+", "_", f"{sistema}_{planeta_b}")
    filas_out, lineas_txt = [], []
    for modo in modos[modo_region]:
        xl, yl = xlims, ylims
        if modo == "referencia":
            res = calcular_kde_panel(d_ref, xl, yl, modo_peso=modo_peso, n_bootstrap=nb,
                                     nivel_min_pico=nivel_min_pico, etiqueta=f"pred_{tipo}_sin_{suf}_{modo_peso}",
                                     cache_dir=cache_dir)
            titulo = "reference sample without the system"
        else:
            anchos = np.asarray(PRED["cov_solo_b_dex"] if cov_solo_b_dex is None else cov_solo_b_dex, dtype=float)
            if anchos.shape != (2,) or not np.isfinite(anchos).all() or (anchos <= 0).any():
                raise ValueError("cov_solo_b_dex debe contener dos desviaciones positivas y finitas en dex.")
            cov, covs, fuente = _calibrar_solo_b(d_ref, d_ext, modo_peso, anchos, nb, avisos)
            nivel = _nivel_isla(100.0) if NIVELES_POR_ALTURA else NIVEL_REGION_ESTABLE
            if nivel is None or not 0 < nivel <= 100:
                raise ValueError("Nivel de contorno inválido.")
            ra, rm = _radios_gauss(cov, nivel)
            xl = (min(xl[0], x_b - ra - 0.02), max(xl[1], x_b + ra + 0.02))
            yl = (min(yl[0], y_b - rm - 0.02), max(yl[1], y_b + rm + 0.02))
            res = calcular_kde_solo_b(x_b, y_b, cov, xl, yl, nivel_min_pico, n_bootstrap=nb,
                                     sigma_a_b=sigma_a_b, sigma_m_b=sigma_m_b, sigma_log_ancho=sigma_log_ancho,
                                     covs_boot=covs, fuente_cov=fuente, cache_dir=cache_dir)
            titulo = "planet b only (1 point)"
        avisos.extend(res.get("avisos", []))
        filas, info = _evaluar_c(res, lx_c, ly_c, rango_ref)
        if res["malla"] is not None and not info["c_en_malla"]:
            avisos.append(f"{modo}: c queda fuera de la malla; densidad interpolada n.d.")
        if any(f.get(f"{k}_acotado_malla") for f in filas for k in LIMITES_REGION):
            avisos.append(f"{modo}: algún borde de isla alcanza el límite de la malla; se interpreta como cota.")
        if not filas:
            filas = [{"Region": np.nan, "c_en_region_nominal": np.nan, "c_en_region_1sigma": np.nan,
                      "c_en_region_3sigma": np.nan, "error_estimado": False, "Estado_error": "sin error estimado",
                      "criterios_coinciden": np.nan, "criterio_decision": "sin región estimable",
                      "criterio_bordes_c": np.nan, "criterio_densidad_c": np.nan,
                      "a_min_UA": np.nan, "a_max_UA": np.nan, "a_min_err": np.nan, "a_max_err": np.nan,
                      "m_min_Mt": np.nan, "m_max_Mt": np.nan, "N_replicas_borde_min": 0}]
            for nombre in LIMITES_REGION:
                unidad = "UA" if nombre.startswith("a_") else "Mt"
                for borde in ("exacto", "malla"):
                    filas[0][f"{nombre}_{borde}_{unidad}"] = np.nan
                for fuente in ("total", *FUENTES_ERROR):
                    for lado in ("menos", "mas"):
                        filas[0][f"{nombre}_{fuente}_err_{lado}_{unidad}"] = np.nan
        if any(not f["error_estimado"] for f in filas):
            avisos.append(f"{modo}: sin error estimado para al menos una región o borde requerido.")
        if any(f.get("criterios_coinciden") is False for f in filas):
            avisos.append(f"{modo}: la densidad y la proyección radial discrepan en al menos una región.")
        if any(f.get("interpolacion_coincide_con_nominal") is False for f in filas):
            avisos.append(f"{modo}: la pertenencia interpolada difiere de la decisión nominal en al menos una región.")
        if METODO_SIGMA_DIBUJO == "percentil" and any(
            5 <= f.get("N_replicas_borde_min", 0) < 10 / ((1 - _cobertura_gauss(3)) / 2) for f in filas
        ):
            avisos.append(f"{modo}: las colas del intervalo 3σ tienen pocas réplicas efectivas.")
        with np.errstate(over="ignore", under="ignore"):
            razon_a = float(np.power(10.0, lx_c - x_b))
            razon_p = float(np.power(10.0, 1.5 * (lx_c - x_b)))
        cov = res.get("cov", np.full((2, 2), np.nan))
        base = {"Modo": modo, "Sistema": sistema, "Tipo_Estrella": tipo, "Planeta_b": planeta_b,
                "a_b_UA": a_b, "m_b_Mt": m_b, "Masa_Estrella_Msol": m_estrella,
                "Num_Planetas": fila_b.get("Num_Planetas", np.nan),
                "Num_Planetas_Leidos": fila_b.get("Num_Planetas_Leidos", np.nan),
                "Fuente_Num_Planetas": fila_b.get("Fuente_Num_Planetas", "n.d."),
                "a_c_UA": a_c, "m_c_Mt": m_c, "a_c_sobre_a_b": razon_a,
                "P_c_sobre_P_b": razon_p, "Pesos": modo_peso, "N_referencia": len(d_ref),
                "N_sistemas_referencia": n_sis_ref, "Fuente_cov_solo_b": res.get("fuente_cov", "n.d."),
                "Fuente_incertidumbre": res.get("fuente_incertidumbre", "n.d."),
                "Ancho_a_dex": float(np.sqrt(cov[0, 0])), "Ancho_m_dex": float(np.sqrt(cov[1, 1])),
                "Cov_a_m_dex2": float(cov[0, 1]), "sigma_a_b_UA": res.get("sigma_a_b"),
                "sigma_m_b_Mt": res.get("sigma_m_b"), "sigma_log_ancho": res.get("sigma_log_ancho"),
                "Observacional_estimado_a": modo == "solo_b" and nb >= 5 and sigma_a_b is not None,
                "Observacional_estimado_m": modo == "solo_b" and nb >= 5 and sigma_m_b is not None,
                "Metodo_intervalo_sigma": METODO_SIGMA_DIBUJO, **info}
        filas_out.extend({**base, **f} for f in filas)
        ruta = base_dir / f"Prediccion_c_{suf}_{modo}_{modo_peso}"
        x_plot = (min(xl[0], lx_c - 0.04), max(xl[1], lx_c + 0.04))
        y_plot = yl if ly_c is None else (min(yl[0], ly_c - 0.04), max(yl[1], ly_c + 0.04))
        _figura_prediccion(res, d_ref, fila_b, sistema, tipo, a_c, m_c, titulo, x_plot, y_plot,
                           str(ruta), cmap, filas)
        _figura_perfil(res, a_b, a_c, f"{sistema} · perfil de densidad ({modo})",
                       str(base_dir / f"Perfil_{suf}_{modo}_{modo_peso}"), filas)
        nominales = [f["c_en_region_nominal"] for f in filas if pd.notna(f["c_en_region_nominal"])]
        veredicto = ("DENTRO" if any(nominales) else "FUERA") if nominales else "n.d."
        lineas = [f"Modo {modo} | c = {a_c:.5g} UA | veredicto nominal {veredicto}",
                  f"Densidad KDE interpolada en c = {_formato_num(info['Densidad_rel_en_c_%'], 4)} % | "
                  f"cobertura del rango de referencia en log(a) = {_formato_num(info['Cobertura_log_a_%'], 4)} %."]
        if modo == "solo_b":
            lineas.extend((
                f"Kernel: {res['fuente_cov']} | σ_log(a) = {base['Ancho_a_dex']:.5g} dex | "
                f"σ_log(m) = {base['Ancho_m_dex']:.5g} dex.",
                f"Semiancho en dex = σ_log(a)·√[−2 ln(nivel/100)]. El peso de b se cancela a pico 100 %.",
                f"Densidad analítica en c = {_formato_num(info['Densidad_analitica_en_c_%'], 4)} % | "
                f"fuentes de error: {base['Fuente_incertidumbre']}.",
                f"σ(a_b) = {_formato_num(sigma_a_b)} UA | σ(m_b) = {_formato_num(sigma_m_b)} Mt.",
                f"Sensibilidad del ancho: σ en log natural del factor = {res['sigma_log_ancho']:.5g}, "
                "cuando no hay bootstrap de la covarianza."))
        for f in filas:
            lineas.extend(_diagnostico_region(f))
        lineas_txt += lineas + [""]
        print("\n".join(lineas[:2]))
    avisos = list(dict.fromkeys(avisos))
    if avisos:
        print("\nResumen de advertencias")
        for aviso in avisos:
            print(f" - {aviso}")
    tabla = pd.DataFrame(filas_out)
    tabla["Advertencias"] = " | ".join(avisos)
    tabla.to_csv(base_dir / f"prediccion_c_{suf}_{modo_peso}.csv", index=False, encoding="utf-8-sig", na_rep="NaN")
    cab = [f"Predicción de c | {sistema} | tipo {tipo} | b = {planeta_b}",
           f"Modo de pesos: {modo_peso}.",
           f"Intervalos 1σ y 3σ: envolventes de bordes marginales ({METODO_SIGMA_DIBUJO}).",
           "Cobertura: fracción bajo una línea base uniforme en log(a) del rango de referencia.", ""]
    (base_dir / f"prediccion_c_{suf}_{modo_peso}.txt").write_text(
        "\n".join(cab + lineas_txt + ["Advertencias"] + avisos) + "\n", encoding="utf-8")
    tabla.attrs["advertencias"] = avisos
    print(f"\nResultados guardados en {base_dir.resolve()}.")
    return tabla


def ejecutar_prediccion_c(cfg=None):
    """Lee los datos y ejecuta la predicción a partir del diálogo o de PRED."""
    p = {**PRED, **(cfg or {})}
    entrada = p["entrada"]
    if entrada not in {"dialogo", "parametros"}:
        raise ValueError(f"entrada desconocida: {entrada!r}.")
    if entrada == "parametros" and not p["carpetas"]:
        raise ValueError("Con entrada='parametros' se requiere al menos una carpeta en PRED['carpetas'].")
    df, base = extraer_datos_v2(p["carpetas"] if entrada == "parametros" else None,
                                catalogo_multiplicidad=p["catalogo_multiplicidad"])
    if df.empty:
        print("No hay planetas válidos para predecir.")
        return None
    sel = _dialogo_objetivo(df) if entrada == "dialogo" else {
        k: p[k] for k in ("sistema", "planeta_b", "a_c", "m_c")}
    if not sel:
        print("Selección cancelada.")
        return None
    if not sel["sistema"] or not sel["planeta_b"] or sel["a_c"] is None:
        raise ValueError("Con entrada='parametros' define sistema, planeta_b y a_c.")
    out = Path(p["salida"] or Path(base or os.getcwd()) / "graficos_prediccion_c")
    out.mkdir(parents=True, exist_ok=True)
    df.to_csv(out / "datos_extraidos.csv", index=False, encoding="utf-8-sig", na_rep="NaN")
    return predecir_c(df, **sel, modo_region=p["modo_region"], modo_peso=p["modo_peso"],
                      n_bootstrap=p["n_bootstrap"], nivel_min_pico=p["nivel_min_pico"],
                      cov_solo_b_dex=p["cov_solo_b_dex"], base_dir=out, sigma_a_b=p["sigma_a_b"],
                      sigma_m_b=p["sigma_m_b"], sigma_log_ancho=p["sigma_log_ancho"])

## Ejecución interactiva o por parámetros

Ejecuta la llamada cuando hayas elegido la configuración en PRED.
La segunda llamada original queda disponible como ejemplo; descoméntala solo
si quieres hacer otra predicción. La ejecución de las pruebas no abre diálogos.

In [ ]:
# Descomenta estas líneas para abrir los diálogos o usar PRED con entrada="parametros".
# resultado = ejecutar_prediccion_c()
# resultado

In [ ]:
# Segunda predicción opcional con una configuración por llamada.
# resultado_2 = ejecutar_prediccion_c({"entrada": "parametros", "carpetas": [...],
#                                    "sistema": "...", "planeta_b": "b", "a_c": 0.0158})
# resultado_2

## Auditoría y decisiones de método

Los números identifican las celdas contando desde cero en el notebook original.
La celda final reproduce la comparación antes/después y los casos límite.

| Celda y función | Síntoma | Causa raíz y evidencia | Severidad | Corrección |
|---|---|---|---|---|
| 17, 27 · _regiones_por_picos, calcular_kde_solo_b | Falso negativo en TRAPPIST‑1 | Los extremos de cuatro vecinos retenían solo nodos. La malla daba a_max ≈ 0,01568 UA; c = 0,0158 UA y su densidad interpolada era ≈ 92,6 %. | Alta | Interpola los cruces de aristas en referencia y usa el elipsoide analítico en solo_b. Conserva ambos conjuntos de bordes para comparación. |
| 17, 19, 27 · _nivel_isla, _fusionar_islas, _dist_norm, _centro_tramo, _evaluar_c | Una proyección radial podía aceptar un punto en otra isla, o fuera del contorno en masa. Islas vecinas podían fusionarse tras detectarse como distintas. | La evaluación ignoraba el umbral local y la fusión usaba solo la proximidad de cajas. La ventana de picos usaba el paso de x para ambos ejes. | Alta | Identidad por componente, nivel estrictamente inferior al pico y ventana en dex por eje. Une la prueba 2D con la proyección y exporta desacuerdos. Se conserva _fusionar_islas como utilidad histórica. |
| 10, 11, 17, 19, 21, 27 · _sigma_log, _intervalo_sigma, _filas_sigma, _error_asimetrico, _txt_lim, _txt_masa, _filas_resumen, _escribir_texto_regiones, calcular_kde_solo_b, _evaluar_c | Cero aparente con pocas réplicas. El flag 3σ copiaba el nominal. Las masas aparecían bajo encabezados de UA. | Faltaban réplicas en solo_b, había retornos a cero o al nominal y etiquetas con redondeo fijo. | Alta | Cinco réplicas finitas como mínimo, NaN con estado explícito, cero admisible con réplicas idénticas, bandas 1σ/3σ calculadas por borde y unidades explícitas. |
| 19, 21 · _bootstrap_regiones, calcular_kde_panel | Bootstrap de filas para uno o dos sistemas y rechazo de réplicas válidas con ancho fijo. | Se descartaban los grupos pequeños y luego se exigía varianza de coordenadas aunque el kernel ya fuera positivo definido. La asignación podía fusionar varias islas en una réplica. | Alta | Un sistema da incertidumbre entre sistemas n.d. Con dos sistemas se remuestrean ambos por bloques. Las réplicas con un único sistema muestreado sí pueden evaluarse con kernel fijo. Asignación uno a uno con tolerancia y bordes subcelda. |
| 27, 29 · calcular_kde_solo_b, predecir_c | En solo_b el respaldo daba ±0,15 dex con independencia de la muestra y sin incertidumbre. | El peso se cancela y el pico de la gaussiana queda fijado a 100 %. El parámetro de respaldo contiene desviaciones en dex. | Alta | Prioriza el kernel del tipo o de todos los tipos disponibles y excluye todo el objetivo. Si no hay dos sistemas con un ajuste 2D válido, advierte y usa el respaldo. Registra el ancho, la fuente y su incertidumbre. |
| 23 · _leer_carpeta_resultados, extraer_datos_v2 | Datos imputados sin evidencia, nombres sensibles a la ruta y zonas o áreas tomadas de líneas incorrectas. | Defaults 0/1, coincidencias parciales, reemplazo global de comas, detección del último archivo y lectura con errores ignorados. | Alta | Etiquetas ancladas y validadas con unidades, campos semánticos de README, parejas únicas, diagnóstico por archivo y descarte explícito. Los conteos opcionales ausentes quedan NaN. |
| 21, 23, 27, 29 · plot_matriz_tipo_multiplicidad, extraer_datos_v2, _cobertura_log_a, predecir_c | Referencia vacía con cobertura ficticia. Lectura de un único planeta le asignaba multiplicidad 1 a TRAPPIST‑1. | Se reutilizaban ejes en lugar del rango observado y se contaban solo los registros leídos. | Alta | Cobertura n.d. sin rango válido. Multiplicidad de README o catálogo local, con columna aparte para planetas leídos. La matriz omite multiplicidades desconocidas. |
| 7, 10, 21 · _safe_log10, _calcular_pesos, _pesos_validos, _kde_cov_fija, _calcular_limites_criticos, _bootstrap_limites_criticos, _umbral_region, calcular_kde_panel | Vacíos, pesos no finitos o KDE singular; prueba de pesos siempre verdadera para el rango. | np.ptp no prueba dispersión útil aquí y falla con un vector vacío. Los defaults de área omitida afectaban pesos y el ajuste no verificaba el rango 2D ponderado. | Media-alta | Valida coordenadas, áreas y pesos; conserva su fórmula. Requiere tres puntos efectivos de rango dos; evita normalizar campos vacíos. El bootstrap histórico conserva su ancho explícito 0,6 y el panel y sus réplicas adaptables usan 0,55. |
| 13, 21, 27, 29 · _set_cache_dir, _hash_cache, _cache_obtener, calcular_kde_panel, calcular_kde_solo_b, predecir_c | Reutilización de la ruta anterior, caché parcialmente escrita y solo_b sin clave. | Estado global pegajoso, excepciones amplias y claves sin los parámetros del caso de un punto. | Media-alta | Rutas explícitas por llamada, claves versionadas con covarianza y remuestreo, manejo acotado de errores y publicación atómica. Usa cachés locales de confianza. |
| 25, 27, 29–32 · _dialogo_objetivo, _densidad_en_c, _intervalo_barra, _fila_planeta, predecir_c, ejecutar_prediccion_c | Cero de masa tratado como ausencia, sufijo ambiguo, densidad 0 fuera de malla y dos ejecuciones con diálogo. | Verificación por truthiness, endswith sin límite de palabra y relleno con 0. La posición de c ampliaba la malla usada para ajustar el modelo. | Media | Masa omitida solo con None, validación positiva finita, sufijo orbital único, densidad n.d. fuera de dominio y malla independiente de c. Las llamadas originales quedan disponibles de forma explícita. |
| 9, 15, 17, 19, 21, 27 · _asignar_estilos_sistema, _cfg_leyenda, _n_entradas_leyenda, _filas_leyenda, _alto_leyenda, _dibujar_isolineas_difuminadas, _limites_por_nivel, _regiones_por_umbral, _limites_masa_region, _masa_region_nominal, _region_principal, dibujar_panel_kde, _dibujar_panel_matriz, _figura_prediccion, _figura_perfil | Límites truncados y perfil sin PDF. Un panel sin referencia no debe mostrar una entrada vacía. | Redondeo, cajas por proyección y soporte de exportación distinto entre figuras. | Media | Se mantiene magma, isolíneas, líneas y rectángulos. Las figuras muestran bordes, fuentes y diagnóstico. Perfil en PNG/PDF. Se mantiene el modo histórico por umbral y su agrupamiento radial. |
| 1, 7, 11 · tqdm, _mostrar_o_cerrar, _cobertura_gauss, _dibujar_bandas_sigma, _guardar_tabla_sigma, _con_sufijo | Cuestiones de ejecución y exportación. | Dependencias opcionales, bandas con pocas réplicas y cuantiles 3σ muy sensibles a las colas. | Baja | Conserva el manejo de dependencias y los helpers. Exporta réplicas efectivas, marca intervalos no estimables y advierte sobre colas 3σ con poco soporte. |

### Propagación y alcance

En solo_b se evalúan por separado los desplazamientos del centro debidos a σ(a_b) y
σ(m_b) y la variación del ancho. Con errores observacionales suministrados se aplica
Monte Carlo lognormal con las coordenadas de b como medianas y la dispersión relativa
especificada. Si falta una σ observacional se informa como n.d. y la propagación se
condiciona al valor nominal de ese eje. No se deducen errores de parámetros redondeados.
Si el ancho viene de la muestra, se remuestrean sistemas y se reajusta la covarianza.
En ausencia de al menos cinco covarianzas válidas se usa la sensibilidad parametrizada
al ancho con ≈95 % de factores entre 0,5× y 2× para el valor de respaldo de PRED.
El total se obtiene aplicando ambos cambios en cada réplica, con semillas independientes
para las fuentes. Las regiones de referencia usan la covarianza nominal en el bootstrap
si KDE_COV_FIJA está activo.

Sensibilidad al ancho sola permite cuantificar la dependencia del modelo cuando faltan
errores de b. Los errores observacionales solos dejan la dependencia del kernel sin
medir. El bootstrap de la covarianza se prefiere cuando la muestra lo permite porque
usa la dispersión y agrupación observadas. La comprobación numérica siguiente compara
ambas fuentes en igualdad de condiciones y usa σ observacionales sintéticas.

Los niveles expresan densidad KDE relativa. Las envolventes 1σ/3σ son marginales y
no calibran una probabilidad conjunta de pertenencia o de estabilidad física. Las
colas 3σ requieren muchas réplicas para estimar cuantiles con precisión. La herencia
entre tipos supone que sus escalas en log(a) y log(m) son transferibles. En regiones
de referencia muy estrechas, la malla puede no resolver un pico. Un borde que toca
la malla se exporta como cota. Las entradas ambiguas del lector requieren un formato
numérico inequívoco. El área total de la grilla se conserva fuera del cálculo de pesos,
que usa la suma de las cuatro áreas especificadas.

### Comprobación con los archivos del proyecto

La lectura de las cuatro carpetas F/G/K/M encontró 451 planetas válidos de 126
sistemas, con tres sistemas de multiplicidad desconocida y cero descartes. Con
solo la carpeta original de TRAPPIST‑1 b, se leyó un planeta y se recuperó la
multiplicidad siete del catálogo. La referencia vacía da 92,62 % y DENTRO con el
kernel de respaldo. Teegarden c queda FUERA nominalmente, a 0,09182 dex del
borde, y entra en la envolvente de sensibilidad a 3σ con ese respaldo.

Con las cuatro carpetas seleccionadas, la referencia M para TRAPPIST‑1 tiene
156 planetas y 46 sistemas. La KDE de referencia da 29,85 % en c y el modo
solo_b con kernel heredado da 86,08 %. Ambos dan FUERA nominalmente bajo esa
muestra. La regresión al 92,62 % corresponde al caso de referencia vacía.

In [ ]:
def probar_prediccion_c():
    """Regresiones reproducibles, sin diálogos ni escrituras persistentes."""
    from contextlib import redirect_stdout
    from io import StringIO
    from unittest.mock import patch

    def esperar_valor_error(func, *args, **kwargs):
        try:
            func(*args, **kwargs)
        except ValueError:
            return
        raise AssertionError(f"Se esperaba ValueError de {func.__name__}.")

    def registro(s, a, m, tipo="M", nombre=None):
        return {"Sistema": s, "Planeta": nombre or f"{s} b", "Tipo_Estrella": tipo,
                "Distancia_UA": a, "Masa_Tierra": m, "Masa_Estrella": 0.1, "Area_Estable_km2": 2.0,
                "Area_Inestable": 4.0, "Area_Colision_m1": 1.0, "Area_Colision_m2": 3.0}

    def ejes(ab):
        return (math.log10(ab) - 0.4, math.log10(5.2) + 0.4), (-0.8, math.log10(317.8) + 0.5)

    def caso_antes(ab, mb, ac, mc, cov, xl, yl):
        # Reproduce los pasos originales para una isla: normalización a máximo de malla,
        # límites por nodos y veredicto radial. La incertidumbre original era 0.
        X, Y = np.mgrid[xl[0]:xl[1]:150j, yl[0]:yl[1]:150j]
        Z = _kde_cov_fija([math.log10(ab)], [math.log10(mb)], [1.0], cov, X, Y)
        Z = 100.0 * (Z / Z.max())
        lab = _label_cc(Z >= 90.9)[0]
        lab_pico = lab[np.unravel_index(Z.argmax(), Z.shape)]
        xs = X[lab == lab_pico]
        lo, hi = float(xs.min()), float(xs.max())
        dens, _ = _densidad_en_c((X, Y, Z), math.log10(ac), math.log10(mc))
        return lo <= math.log10(ac) <= hi, 10.0 ** lo, 10.0 ** hi, 0.0, 0.0, dens

    cov = np.diag(np.square((0.35, 0.80)))
    comparacion = []
    for s, ab, mb, ac, mc, esperado in (
        ("TRAPPIST-1", 0.01154, 1.374, 0.0158, 1.308, True),
        ("Teegarden", 0.0259, 1.16, 0.0455, 1.05, False),
    ):
        xl, yl = ejes(ab)
        antes = caso_antes(ab, mb, ac, mc, cov, xl, yl)
        res = calcular_kde_solo_b(math.log10(ab), math.log10(mb), cov, xl, yl, n_bootstrap=1000)
        filas, info = _evaluar_c(res, math.log10(ac), math.log10(mc), None)
        f = filas[0]
        assert antes[0] is False and f["c_en_region_nominal"] is esperado
        assert f["criterios_coinciden"] and f["umbral_isla_%"] == 90.9
        assert f["a_min_UA"] < f["a_min_malla_UA"] < f["a_max_malla_UA"] < f["a_max_UA"]
        assert f["error_estimado"] and f["a_min_err"] > 0 and f["a_max_err"] > 0
        assert np.isnan(f["a_max_observacional_err_mas_UA"]) and np.isnan(info["Cobertura_log_a_%"])
        assert np.isclose(f["delta_dex_al_borde"], max(0, math.log10(ac / f["a_max_UA"])))
        assert round(info["Densidad_rel_en_c_%"], 1) == (92.6 if esperado else 78.2)
        assert (info["Densidad_rel_en_c_%"] >= 90.9) is esperado
        if esperado:
            assert antes[2] < ac < f["a_max_UA"]
        else:
            assert 0.09 < f["delta_dex_al_borde"] < 0.10
        comparacion.extend([
            (s, "antes", *antes),
            (s, "después", esperado, f["a_min_UA"], f["a_max_UA"], f["a_min_err"], f["a_max_err"],
             info["Densidad_rel_en_c_%"]),
        ])

    # Sin masa: proyección. Una esquina de la caja con masa queda fuera de la elipse.
    res = calcular_kde_solo_b(0, 0, cov, (-1, 1), (-1, 1), n_bootstrap=4)
    ra, rm = _radios_gauss(cov, 90.9)
    assert _evaluar_c(res, 0.9 * ra, None, None)[0][0]["c_en_region_nominal"]
    esquina = _evaluar_c(res, 0.9 * ra, 0.9 * rm, None)[0][0]
    assert not esquina["c_en_region_nominal"] and not esquina["criterios_coinciden"]
    assert np.isnan(esquina["c_en_region_3sigma"]) and np.isnan(esquina["c_en_region_1sigma"])
    assert not esquina["error_estimado"]
    fuera, info = _evaluar_c(res, 2, None, None)
    assert not fuera[0]["c_en_region_nominal"] and np.isnan(info["Densidad_rel_en_c_%"])
    assert not info["c_en_malla"] and info["Masa_log_donde_max"] is None

    # Incertidumbre sin datos, insuficiente y degenerada. En 3σ usa la réplica constante,
    # aunque su valor difiera del nominal.
    pocas = [0.1, 0.2, 0.3, np.nan, np.inf]
    assert np.isnan(_sigma_log(pocas)) and np.isnan(_error_asimetrico(1, pocas)).all()
    assert _intervalo_sigma(pocas, 3) is None
    iguales = [0.13] * 5
    assert _sigma_log(iguales, "std") == _sigma_log(iguales, "percentil") == 0
    assert _error_asimetrico(1, iguales) == (0, 0)
    assert _intervalo_sigma(iguales, 3, central=0, metodo="std") == (0.13, 0.13)
    fs = _filas_sigma({}, {"m_min_1": (0.0, pocas), "a_min_1": (0.13, iguales)}, ks=[3])
    assert all(f["Unidad"] == "M_Tierra" and not f["error_estimado"] and np.isnan(f["Inf"]) for f in fs[:2])
    assert all(f["Sigma_std"] == 0 and f["error_estimado"] for f in fs[2:])
    cero = calcular_kde_solo_b(0, 0, cov, (-1, 1), (-1, 1), n_bootstrap=0)
    assert np.isnan(_evaluar_c(cero, 0, 0, None)[0][0]["c_en_region_3sigma"])

    # Dos islas apiladas, mismo rango radial: la densidad de una no valida la otra.
    X, Y = np.mgrid[-1:1:121j, -2:2:161j]
    Z = 100 * (np.exp(-(X / .12) ** 2 - ((Y - 1) / .24) ** 2)
               + .82 * np.exp(-(X / .12) ** 2 - ((Y + 1) / .24) ** 2))
    muestra = {"malla": (X, Y, Z), "regiones": [], "boot": {}}
    for t, d in _regiones_por_picos(Z, X, Y, detalles=True):
        _registrar_region(muestra, t, d, {"total": {}})
    fs, _ = _evaluar_c(muestra, 0, 1, (-1, 1))
    assert len(fs) == 2 and sum(f["c_en_region_nominal"] for f in fs) == 1
    assert all(f["criterio_bordes_c"] and f["c_supera_umbral_interpolado"] for f in fs)
    assert any(f["criterios_coinciden"] is False for f in fs)
    assert _nivel_isla(90.9) == 80.8
    assert _regiones_por_picos(np.zeros_like(Z), X, Y) == []

    # Cambios subcelda detectables cuando el índice del borde es el mismo.
    Xg, Yg = np.mgrid[-1:1:81j, -1:1:81j]
    desplazados = []
    for dx in np.linspace(0.0002, 0.0008, 9):
        Zg = 100 * np.exp(-0.5 * (((Xg - dx) / .35) ** 2 + (Yg / .8) ** 2))
        [(t, geom)] = _regiones_por_picos(Zg, Xg, Yg, detalles=True)
        desplazados.append((t[1], geom["nodos"][1]))
    assert np.ptp([n for _, n in desplazados]) == 0
    assert _sigma_log([exacto for exacto, _ in desplazados]) > 0

    b = registro("Objetivo", 0.01154, 1.374)
    otros = [registro("Solo sistema", a, m) for a, m in ((0.1, 1), (0.2, 4), (0.4, 2))]
    d_uno = pd.DataFrame(otros)
    assert _calcular_pesos(d_uno, "fraccion4").tolist() == [0.2] * 3
    vacio = calcular_kde_panel(d_uno.iloc[:0], (-2, 0), (-1, 1), n_bootstrap=8, cache_dir=None)
    assert vacio["malla"] is None and not vacio["regiones"]
    pesos_cero = d_uno.assign(Area_Estable_km2=0)
    assert not _pesos_validos([]) and not _pesos_validos([0, 0]) and not _pesos_validos([1, -1])
    assert _pesos_validos([1, 1]) and not _pesos_validos([np.inf])
    assert calcular_kde_panel(pesos_cero, (-2, 0), (-1, 1), n_bootstrap=8, cache_dir=None)["malla"] is None
    uno = calcular_kde_panel(d_uno, (-2, 0), (-1, 1), n_bootstrap=8, cache_dir=None)
    assert uno["malla"] is not None and uno["regiones"] and all(not v for _, v in uno["boot"].values())
    assert np.isnan(_evaluar_c(uno, -1, 0, (-2, 0))[0][0]["c_en_region_3sigma"])
    d_dos = d_uno.copy()
    d_dos.loc[2, "Sistema"] = "Segundo sistema"
    dos = calcular_kde_panel(d_dos, (-2, 0), (-1, 1), n_bootstrap=16, cache_dir=None)
    assert any(len(v) >= 5 for _, v in dos["boot"].values())
    assert dos["fuente_incertidumbre"] == "bootstrap_sistemas"

    nombres = pd.DataFrame([registro("X", 1, 1, nombre=p) for p in ("X b", "X ab", "X B b")])
    assert _fila_planeta(nombres, "X", "X_b")["Planeta"] == "X b"
    esperar_valor_error(_fila_planeta, nombres, "X", "b")
    assert _fila_planeta(nombres.iloc[:2], "X", "b")["Planeta"] == "X b"
    assert _fila_planeta(nombres, "X", "Bb") is None
    assert _numero_archivo("1.234,56") == _numero_archivo("1,234.56") == 1234.56
    assert _numero_archivo("1 234,56") == 1234.56 and _numero_archivo("1,234e3") == 1234
    esperar_valor_error(_numero_archivo, "1,234")
    esperar_valor_error(predecir_c, pd.DataFrame([b]), "Objetivo", "b", 0.1, 0)
    esperar_valor_error(predecir_c, pd.DataFrame([b]), "Objetivo", "b", np.inf)

    # Prueba numérica de fuentes con incertidumbres inventadas solo para esta simulación.
    ab, mb, n = 0.01154, 1.374, 4096
    xl, yl = ejes(ab)
    mixto = calcular_kde_solo_b(math.log10(ab), math.log10(mb), cov, xl, yl, n_bootstrap=n,
                               sigma_a_b=0.01 * ab, sigma_m_b=0.05 * mb)
    fuentes = mixto["regiones"][0]["fuentes"]
    s_obs, s_ancho, s_total = (_sigma_log(fuentes[k]["a_max"], "std")
                               for k in ("observacional", "ancho", "total"))
    assert 0.0039 < s_obs < 0.0048 and 0.05 < s_ancho < 0.075 and s_total > 0
    assert np.isclose(s_total, math.hypot(s_obs, s_ancho), rtol=0.03)
    parcial = calcular_kde_solo_b(0, 0, cov, (-1, 1), (-1, 1), n_bootstrap=16, sigma_a_b=0.01)
    fp = _evaluar_c(parcial, 0, 0, None)[0][0]
    assert fp["a_max_observacional_err_mas_UA"] > 0 and np.isnan(fp["m_max_observacional_err_mas_Mt"])

    englobado = pd.concat([pd.DataFrame([b]), d_dos.assign(Tipo_Estrella="K")], ignore_index=True)
    # Entradas y salidas reales, rutas de caché diferentes y catálogo de carpeta parcial.
    with tempfile.TemporaryDirectory() as temp, redirect_stdout(StringIO()) as consola, \
            patch.dict(globals(), {"MOSTRAR_FIGURAS": False, "USAR_CACHE_KDE": True}):
        tmp = Path(temp)
        ref = tmp / "ref"
        solo = tmp / "solo"
        d_b = pd.DataFrame([b])
        tabla_ref = predecir_c(d_b, "Objetivo", "b", 0.02, modo_region="referencia", base_dir=ref, n_bootstrap=0)
        assert tabla_ref["N_referencia"].iloc[0] == 0 and tabla_ref["Cobertura_log_a_%"].isna().all()
        assert tabla_ref["c_en_region_3sigma"].isna().all() and not tabla_ref["error_estimado"].any()
        t = predecir_c(d_b, "Objetivo", "b", 0.013, base_dir=solo, modo_region="solo_b", n_bootstrap=0)
        assert t["c_en_region_nominal"].iloc[0] and t["c_en_region_3sigma"].isna().all()
        assert "sin error estimado" in next(solo.glob("prediccion*.txt")).read_text(encoding="utf-8")
        assert "NaN" in next(solo.glob("prediccion*.csv")).read_text(encoding="utf-8-sig")
        assert next(solo.glob("Perfil*.pdf")).is_file() and next(solo.glob("Perfil*.png")).is_file()
        assert len(list((ref / CACHE_KDE_NOMBRE).glob("*.pkl"))) == 1
        pkl = next((solo / CACHE_KDE_NOMBRE).glob("*.pkl"))
        pkl.write_bytes(b"cache simulada ilegible")
        t2 = predecir_c(d_b, "Objetivo", "b", 100.0, base_dir=solo, modo_region="solo_b", n_bootstrap=0)
        assert (t["a_max_UA"] == t2["a_max_UA"]).all() and not t2["c_en_malla"].iloc[0]
        assert "caché ilegible" in consola.getvalue()
        t3 = predecir_c(d_b, "Objetivo", "b", 0.013, base_dir=solo, modo_region="solo_b",
                        n_bootstrap=0, cov_solo_b_dex=(0.45, 0.80))
        assert t3["a_max_UA"].iloc[0] > t["a_max_UA"].iloc[0]
        assert len(list((solo / CACHE_KDE_NOMBRE).glob("*.pkl"))) == 2
        cov_ext, covs, origen = _calibrar_solo_b(
            englobado.iloc[:0], englobado.iloc[1:], "fraccion4", (0.35, 0.80), 32, [])
        assert origen == "referencia de todos los tipos" and len(covs) >= 5
        heredado = calcular_kde_solo_b(0, 0, cov_ext, (-1, 1), (-1, 1), n_bootstrap=32, covs_boot=covs)
        assert "bootstrap_sistemas" in heredado["regiones"][0]["fuentes"]
        carpeta = tmp / "nombre de carpeta ajeno"
        carpeta.mkdir()
        rd = carpeta / "README_b.txt"
        rr = carpeta / "b_resumen.txt"
        rd.write_text("SISTEMA: Prueba catálogo\nPLANETA: Prueba catálogo b\n"
                      "Semi-eje mayor de la órbita (UA): 0.01154\n"
                      "Masa del Planeta (Masas Terrestres): 1.374\n", encoding="utf-8")
        rr.write_text("Área estable: 1,200.5 km²\nÁrea inestable: 2.000,75 km²\n"
                      "Área de colisión con m1: 0 km²\nÁrea de colisión con m2: 20 km²\n",
                      encoding="utf-8")
        cat = tmp / "estrellas.csv"
        cat.write_text("Host_Name,Number_of_Planets\nPrueba catálogo,7\n", encoding="utf-8")
        leidos, _ = extraer_datos_v2([("M", carpeta)], catalogo_multiplicidad=cat)
        assert leidos["Sistema"].iloc[0] == "Prueba catálogo" and leidos["Num_Planetas"].iloc[0] == 7
        assert leidos["Num_Planetas_Leidos"].iloc[0] == 1 and leidos["Masa_Estrella"].isna().all()
        assert leidos["Area_Estable_km2"].iloc[0] == 1200.5
        rr.write_text("Área extra de colisión con m1: 12 km²\nÁrea estable: 2 km²\n"
                      "Área inestable: 3 km²\nÁrea de colisión con m2: 0 km²\n", encoding="utf-8")
        malo, _ = extraer_datos_v2([("M", carpeta)], catalogo_multiplicidad=cat)
        assert malo.empty and len(malo.attrs["diagnostico_lectura"]["descartes"]) == 1
        assert str(rr) in malo.attrs["diagnostico_lectura"]["descartes"][0]

    tabla = pd.DataFrame(comparacion, columns=["Sistema", "Versión", "Dentro nominal", "a_min UA", "a_max UA",
                                               "Error medio a_min UA", "Error medio a_max UA", "Densidad en c %"])
    print(tabla.to_string(index=False, float_format=lambda x: f"{x:.6g}"))
    print(f"\nPrueba sintética · dispersión de log(a_max): observacional {s_obs:.5f} dex, "
          f"ancho {s_ancho:.5f} dex, total {s_total:.5f} dex.")
    print("Pruebas superadas: regresiones astronómicas, geometría, incertidumbre, muestras límite, "
          "lectura, multiplicidad, caché y salidas.")
    return tabla


resultados_pruebas = probar_prediccion_c()

   Sistema Versión  Dentro nominal   a_min UA  a_max UA  Error medio a_min UA  Error medio a_max UA  Densidad en c %
TRAPPIST-1   antes           False 0.00826388 0.0156799                     0                     0          92.6409
TRAPPIST-1 después            True  0.0081155 0.0164095            0.00107096            0.00230166          92.6248
 Teegarden   antes           False  0.0183308 0.0358683                     0                     0          78.1991
 Teegarden después           False  0.0182142  0.036829            0.00240362            0.00516577          78.1924

Prueba sintética · dispersión de log(a_max): observacional 0.00436 dex, ancho 0.05903 dex, total 0.05915 dex.
Pruebas superadas: regresiones astronómicas, geometría, incertidumbre, muestras límite, lectura, multiplicidad, caché y salidas.
